In [0]:
%pip install "numpy<2.0" faiss-cpu==1.8.0 langgraph==0.2.27 langchain==0.3.7 --quiet
dbutils.library.restartPython()

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 06 — LangGraph Healthcare Intelligence Orchestrator (v5.1 — Bug-Fixed)
# MAGIC
# MAGIC ## Critical Fixes vs v5
# MAGIC | # | Bug | Fix |
# MAGIC |---|-----|-----|
# MAGIC | 1 | SQL uses bare `gold_*` table names → `TABLE_OR_VIEW_NOT_FOUND` | `_qualify_table_names()` enforces `virtue_foundation.ghana.*` in every generated query |
# MAGIC | 2 | `run_agent` returns `None` → `invoke` exception swallowed by MLflow context manager | Wrapped `invoke` in `try/except`; unconditional `return` at end of function |
# MAGIC | 3 | `ensure_list` numpy DeprecationWarning on empty-array truth test | Explicit `len()` check before truth test |
# MAGIC | 4 | Node-level `mlflow.start_run(nested=True)` crashes without active parent | All nested runs wrapped in `try/except` |
# MAGIC | 5 | `_route_after_node` can return a key not in the edge dict | Guard returns `"synthesiser"` for any unknown agent name |
# MAGIC | 6 | `AgentStateV2.steps` reducer causes None accumulation on early failures | Every node explicitly spreads `**state` before overriding fields |
# MAGIC | 7 | `gold_anomaly_flags` schema missing `enhanced_graph_dependency_gap` | Column guard in `anomaly_node` via `get_table_cols` intersection |
# MAGIC | 8 | `planning_node` references `plan_text` outside scope on error | Variable scoped before the try block |

# COMMAND ----------
# MAGIC %pip install "numpy<2.0" faiss-cpu==1.8.0 langgraph==0.2.27 langchain==0.3.7 --quiet

# COMMAND ----------
# MAGIC %md ## 0. Imports & Configuration

# COMMAND ----------

import json, os, re, time, math, pickle
import operator
from typing import (TypedDict, Annotated, List, Optional, Dict, Any, Tuple,
                    Set, NamedTuple)
from datetime import datetime, timezone
from enum import Enum

import numpy as np
import pandas as pd
import requests
import mlflow
import faiss

from langgraph.graph import StateGraph, END
from pyspark.sql import SparkSession, functions as F, types as T

spark = SparkSession.builder.getOrCreate()
print(f"Python env ready | {datetime.now(timezone.utc).isoformat()}")

# COMMAND ----------

class AgentConfig:
    DATABRICKS_HOST: str = spark.conf.get("spark.databricks.workspaceUrl", "").rstrip("/")

    @staticmethod
    def _get_token() -> str:
        for fn in [
            lambda: dbutils.secrets.get(scope="virtue-foundation", key="databricks-token"),
            lambda: dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get(),
            lambda: os.environ.get("DATABRICKS_TOKEN", ""),
        ]:
            try:
                t = fn()
                if t:
                    return t
            except Exception:
                pass
        raise RuntimeError("No Databricks token found.")

    LLM_MODEL   = "databricks-meta-llama-3-3-70b-instruct"
    EMBED_MODEL  = "databricks-bge-large-en"
    EMBED_DIM    = 1024

    # ── Fully-qualified table names (CRITICAL: always use these) ──────────────
    CATALOG      = "virtue_foundation.ghana"
    IDP_TABLE        = "virtue_foundation.ghana.gold_idp_enriched"
    FACILITIES_TABLE = "virtue_foundation.ghana.gold_facilities_enriched"
    REGIONAL_TABLE   = "virtue_foundation.ghana.gold_regional_summary"
    DESERT_TABLE     = "virtue_foundation.ghana.gold_medical_desert_scores"
    ANOMALY_FLAGS    = "virtue_foundation.ghana.gold_anomaly_flags"
    ANOMALY_REPORT   = "virtue_foundation.ghana.gold_anomaly_report"
    PRIORITY_TABLE   = "virtue_foundation.ghana.gold_regional_priority"

    RAG_DIR      = "/Workspace/Users/dasdeepayan08@gmail.com/databricks_accenture_hackathon_virtue_foundationtrack/databricks/rag"
    VOLUME_INDEX = "/Volumes/virtue_foundation/ghana/rag/faiss_index.bin"
    VOLUME_META  = "/Volumes/virtue_foundation/ghana/rag/faiss_metadata.json"

    MLFLOW_EXP = "/Users/dasdeepayan08@gmail.com/virtue-foundation-idp-hackathon"

    GHANA_CITY_COORDS: Dict[str, Tuple[float, float]] = {
        "accra":       (5.5600, -0.1969), "tema":      (5.6698, -0.0166),
        "kumasi":      (6.6884, -1.6244), "takoradi":  (4.8978, -1.7561),
        "tamale":      (9.4035, -0.8393), "cape coast":(5.1053, -1.2466),
        "ho":          (6.6008,  0.4703), "koforidua": (6.0844, -0.2554),
        "sunyani":     (7.3349, -2.3272), "wa":        (10.0601,-2.5099),
        "bolgatanga": (10.7856, -0.8514), "techiman":  (7.5811, -1.9356),
        "berekum":     (7.4544, -2.5857), "nkawkaw":   (6.5383, -0.7673),
        "obuasi":      (6.2000, -1.6667), "tarkwa":    (5.3000, -1.9833),
        "sekondi":     (4.9374, -1.7073), "winneba":   (5.3525, -0.6267),
        "navrongo":   (10.8942, -1.0940), "bawku":     (11.059, -0.2428),
        "lawra":      (10.6547, -2.9064), "damongo":   (9.0802, -1.8215),
        "salaga":      (8.5575, -0.5154), "nkoranza":  (7.5667, -1.7167),
        "goaso":       (6.7971, -2.5191), "savelugu":  (9.6246, -0.8328),
        "yendi":       (9.4427, -0.0081), "hohoe":     (7.1528,  0.4731),
        "dunkwa":      (5.9651, -1.7738), "sefwi":     (6.5000, -2.4167),
    }


cfg = AgentConfig()
cfg.DATABRICKS_TOKEN = AgentConfig._get_token()
cfg.LLM_ENDPOINT    = f"https://{cfg.DATABRICKS_HOST}/serving-endpoints/{cfg.LLM_MODEL}/invocations"
cfg.EMBED_ENDPOINT  = f"https://{cfg.DATABRICKS_HOST}/serving-endpoints/{cfg.EMBED_MODEL}/invocations"
cfg.RAG_INDEX_PATH  = f"{cfg.RAG_DIR}/faiss_index.bin"
cfg.RAG_META_PATH   = f"{cfg.RAG_DIR}/faiss_metadata.json"

# ── Short → fully-qualified table name map (used in SQL post-processing) ──────
_SHORT_TO_FULL: Dict[str, str] = {
    "gold_idp_enriched":          cfg.IDP_TABLE,
    "gold_facilities_enriched":   cfg.FACILITIES_TABLE,
    "gold_regional_summary":      cfg.REGIONAL_TABLE,
    "gold_medical_desert_scores": cfg.DESERT_TABLE,
    "gold_anomaly_flags":         cfg.ANOMALY_FLAGS,
    "gold_anomaly_report":        cfg.ANOMALY_REPORT,
    "gold_regional_priority":     cfg.PRIORITY_TABLE,
}

print(f"LLM  endpoint : {cfg.LLM_ENDPOINT}")
print(f"Token         : {'✅ set' if cfg.DATABRICKS_TOKEN else '❌ MISSING'}")

# COMMAND ----------
# MAGIC %md ## 1. Query Types & Enums

# COMMAND ----------

class QueryType(str, Enum):
    FACILITY_LOOKUP       = "facility_lookup"
    FACILITY_COMPARISON   = "facility_comparison"
    REGIONAL_ANALYSIS     = "regional_analysis"
    DESERT_ANALYSIS       = "desert_analysis"
    ANOMALY_ANALYSIS      = "anomaly_analysis"
    CAPABILITY_VALIDATION = "capability_validation"
    GEO_SEARCH            = "geo_search"
    SPECIALTY_GAP         = "specialty_gap_analysis"
    HEALTHCARE_PLANNING   = "healthcare_planning"
    RISK_ASSESSMENT       = "risk_assessment"
    INFRASTRUCTURE        = "infrastructure_analysis"
    NGO_COORDINATION      = "ngo_coordination"
    CAPABILITY_GRAPH      = "capability_graph_reasoning"
    PRIORITY_ANALYSIS     = "priority_analysis"
    GENERAL               = "general"


class NodeStatus(str, Enum):
    SUCCESS  = "success"
    PARTIAL  = "partial"
    FALLBACK = "fallback"
    ERROR    = "error"

# COMMAND ----------
# MAGIC %md ## 2. Canonical Agent State

# COMMAND ----------

class EvidenceBundle(TypedDict):
    source_table     : str
    source_row_ids   : List[str]
    columns_used     : List[str]
    confidence       : float
    reasoning_summary: str
    raw_support      : List[Dict]


class NodePayload(TypedDict):
    node_name : str
    status    : str
    summary   : str
    confidence: float
    results   : List[Dict]
    evidence  : List[Dict]
    timing_ms : int


class AgentStateV2(TypedDict):
    query           : str
    session_id      : str
    query_id        : str
    chat_history    : List[Dict]
    query_type      : str
    query_complexity: str
    execution_plan  : Dict
    sub_agents      : List[str]
    sql_results     : Optional[str]
    sql_row_count   : int
    rag_results     : List[Dict]
    geo_results     : Optional[Dict]
    map_data        : Optional[Dict]
    anomaly_results : List[Dict]
    desert_top      : List[Dict]
    desert_data     : Optional[Dict]
    graph_results   : List[Dict]
    priority_data   : Optional[Dict]
    medical_analysis: Optional[Dict]
    planning_data   : Optional[Dict]
    ngo_data        : Optional[Dict]
    evidence_bundles: List[EvidenceBundle]
    citations       : List[Dict]
    step_citations  : List[Dict]
    confidence_score    : float
    hallucination_risk  : float
    retrieval_quality   : float
    answer           : str
    structured_answer: Dict
    warnings  : List[str]
    errors    : List[str]
    steps     : Annotated[List[str], operator.add]
    node_payloads: List[NodePayload]
    run_id    : str

# COMMAND ----------
# MAGIC %md ## 3. Shared Utilities

# COMMAND ----------

def call_llama(messages: List[Dict], system_prompt: Optional[str] = None,
               max_tokens: int = 1024, temperature: float = 0.1,
               retries: int = 3) -> str:
    full: List[Dict] = []
    if system_prompt:
        full.append({"role": "system", "content": system_prompt})
    full.extend(messages)
    headers = {
        "Authorization": f"Bearer {cfg.DATABRICKS_TOKEN}",
        "Content-Type":  "application/json",
    }
    payload = {"messages": full, "max_tokens": max_tokens,
                "temperature": temperature, "top_p": 0.95, "stream": False}
    for attempt in range(retries):
        try:
            r = requests.post(cfg.LLM_ENDPOINT, headers=headers,
                              json=payload, timeout=120)
            r.raise_for_status()
            return r.json()["choices"][0]["message"]["content"]
        except requests.HTTPError as e:
            code = getattr(e.response, "status_code", None)
            if code == 429:
                wait = 5 * (2 ** attempt)
                print(f"  Rate-limit: waiting {wait}s")
                time.sleep(wait)
            elif attempt == retries - 1:
                return f"[LLM error {code}: {str(e)[:150]}]"
            else:
                time.sleep(3 * (attempt + 1))
        except Exception as e:
            if attempt == retries - 1:
                return f"[LLM error: {str(e)[:150]}]"
            time.sleep(3 * (attempt + 1))
    return ""


def parse_json_llm(raw: str, fallback: Optional[Dict] = None) -> Dict:
    if not raw:
        return fallback or {}
    clean = re.sub(r"^```(?:json)?\s*", "", raw.strip(), flags=re.IGNORECASE)
    clean = re.sub(r"\s*```\s*$", "", clean).strip()
    s, e  = clean.find("{"), clean.rfind("}")
    if s != -1 and e != -1:
        clean = clean[s: e + 1]
    for fix in [
        lambda x: x,
        lambda x: re.sub(r",\s*([}\]])", r"\1", x),
        lambda x: re.sub(r"'", '"', x),
    ]:
        try:
            return json.loads(fix(clean))
        except Exception:
            pass
    return fallback or {}


def ensure_list(x) -> List[str]:
    """FIX: explicit len() check avoids numpy DeprecationWarning on truth test."""
    if x is None:
        return []
    # numpy arrays: check len explicitly, never use truth value
    if hasattr(x, "__len__") and not isinstance(x, (str, list, tuple, dict)):
        try:
            if len(x) == 0:
                return []
            return [str(v) for v in x if v is not None
                    and str(v).strip() not in ("", "null", "nan", "None")]
        except Exception:
            pass
    if isinstance(x, float) and x != x:   # NaN
        return []
    if isinstance(x, (list, tuple)):
        return [str(v) for v in x if v is not None
                and str(v).strip() not in ("", "null", "nan", "None")]
    if isinstance(x, str):
        s = x.strip()
        if s in ("", "null", "[]", "nan", "None", "NaN"):
            return []
        try:
            p = json.loads(s)
            if isinstance(p, list):
                return [str(v) for v in p if v]
            return [str(p)]
        except Exception:
            if "|" in s:
                return [v.strip() for v in s.split("|") if v.strip()]
            return [s]
    return [str(x)]


def safe_str(v, d: str = "") -> str:
    if v is None:
        return d
    s = str(v).strip()
    return d if s in ("None", "nan", "null", "", "NaN") else s


def safe_float(v, d: float = 0.0) -> float:
    try:
        if v is None or str(v) in ("nan", "None", "null", "", "NaN"):
            return d
        return float(v)
    except Exception:
        return d


def safe_int(v, d: int = 0) -> int:
    try:
        if v is None or str(v) in ("nan", "None", "null", "", "NaN"):
            return d
        return int(float(v))
    except Exception:
        return d


def haversine_km(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = (math.sin(dlat / 2) ** 2
         + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2)
    return R * 2 * math.asin(math.sqrt(min(a, 1.0)))


def table_exists(name: str) -> bool:
    try:
        spark.table(name).limit(1).collect()
        return True
    except Exception:
        return False


def get_table_cols(name: str) -> Set[str]:
    try:
        return set(spark.table(name).columns)
    except Exception:
        return set()


def _node_payload(
    node_name: str, status: NodeStatus,
    summary: str, confidence: float,
    results: List[Dict], evidence: List[Dict],
    timing_ms: int = 0,
) -> NodePayload:
    return NodePayload(
        node_name=node_name, status=status.value,
        summary=summary, confidence=confidence,
        results=results, evidence=evidence,
        timing_ms=timing_ms,
    )


def _evidence_bundle(
    source_table: str, row_ids: List[str],
    cols: List[str], confidence: float,
    summary: str, raw: List[Dict],
) -> EvidenceBundle:
    return EvidenceBundle(
        source_table=source_table, source_row_ids=row_ids,
        columns_used=cols, confidence=confidence,
        reasoning_summary=summary, raw_support=raw,
    )


def _step_cite(
    step_name: str, input_data: str,
    output_data: str, data_sources: List[str],
    confidence: float,
) -> Dict:
    return {
        "step_name":    step_name,
        "input_data":   input_data[:300],
        "output_data":  output_data[:300],
        "data_sources": data_sources,
        "confidence":   confidence,
    }


def _mlflow_nested(run_name: str):
    """
    FIX: Safe nested MLflow run factory.
    Returns a context manager that silently no-ops if MLflow isn't available
    or there's no active parent run.
    """
    import contextlib

    @contextlib.contextmanager
    def _noop():
        yield None

    try:
        # Check there's an active run before nesting
        if mlflow.active_run() is None:
            return _noop()
        return mlflow.start_run(nested=True, run_name=run_name)
    except Exception:
        return _noop()

# COMMAND ----------
# MAGIC %md ## 4. SQL Schema Registry

# COMMAND ----------

SCHEMA_REGISTRY: Dict[str, Dict] = {
    "gold_idp_enriched": {
        "key": "unique_id",
        "region_col": "region_normalised",
        "anomaly_count_col": "total_stat_anomalies",
        "desert_label_col": "desert_label",
        "cols": {
            "unique_id", "name", "source_url", "pk_unique_id",
            "city_clean", "region_normalised", "latitude", "longitude",
            "address_line1", "address_city", "postal_address", "has_physical_address",
            "facilityTypeId", "operatorTypeId", "facility_type_clean", "facility_type_clean_pdf",
            "organization_type_clean", "organization_category", "ownership_model",
            "is_ngo", "is_faith_based", "is_government", "is_public", "is_private",
            "is_hospital", "is_clinic", "is_teaching_hospital", "is_referral_center",
            "is_military_hospital", "is_specialist_hospital",
            "number_doctors_int", "capacity_int", "area_int", "year_established_int",
            "accepts_volunteers_bool", "ngo_serves_ghana",
            "has_emergency_medicine", "has_obstetrics", "has_surgery", "has_pediatrics",
            "has_icu", "has_radiology", "has_infectious_disease", "has_mental_health",
            "procedure_count", "equipment_count", "capability_count", "specialty_count",
            "specialty_direct_count", "specialty_inferred_count",
            "has_procedures", "has_equipment", "has_capabilities", "has_specialties",
            "is_rag_ready", "is_search_ready", "is_planning_ready", "is_clinical_ready",
            "data_completeness_score", "rag_quality_score", "doc_text",
            "geo_quality_score", "evidence_weight", "clinical_completeness",
            "total_stat_anomalies", "ghost_probability_score", "ghost_review_priority",
            "quality_risk_score", "clinical_risk_score", "operational_risk_score",
            "integrity_risk_score", "capability_is_valid", "capability_confidence",
            "stat_anomaly_capability_inflation", "stat_anomaly_hospital_no_doctors",
            "stat_anomaly_clinic_claims_icu", "stat_anomaly_ghost_facility",
            "stat_anomaly_specialty_mismatch", "stat_anomaly_procedure_breadth",
            "medical_desert_score", "desert_label",
            "emergency_readiness_score", "critical_care_score",
            "capability_graph_nodes", "capability_graph_edges",
            "capability_dependency_gaps", "capability_graph_summary",
            "service_richness_score", "infrastructure_completeness_score",
            "referral_complexity_score", "healthcare_maturity_score",
            "procedure_enriched", "equipment_enriched",
            "capability_enriched", "specialties_enriched",
            "capability_anomalies", "idp_citations", "idp_run_id",
            "specialties_parsed", "procedure_parsed", "equipment_parsed", "capability_parsed",
        },
    },
    "gold_facilities_enriched": {
        "key": "unique_id",
        "region_col": "region_normalised",
        "anomaly_count_col": "total_stat_anomalies",
        "cols": {
            "unique_id", "name", "source_url",
            "city_clean", "region_normalised", "latitude", "longitude",
            "facilityTypeId", "operatorTypeId", "facility_type_clean",
            "organization_type_clean", "organization_category",
            "is_ngo", "is_faith_based", "is_government", "is_public", "is_private",
            "is_hospital", "is_clinic", "is_teaching_hospital", "is_referral_center",
            "number_doctors_int", "capacity_int",
            "accepts_volunteers_bool", "ngo_serves_ghana",
            "has_emergency_medicine", "has_obstetrics", "has_surgery", "has_pediatrics",
            "has_icu", "has_radiology", "has_infectious_disease", "has_mental_health",
            "procedure_count", "equipment_count", "capability_count", "specialty_count",
            "is_rag_ready", "is_search_ready", "is_planning_ready", "is_clinical_ready",
            "data_completeness_score", "rag_quality_score",
            "total_stat_anomalies", "ghost_probability_score",
            "quality_risk_score", "clinical_risk_score", "integrity_risk_score",
            "capability_is_valid",
            "medical_desert_score", "desert_label",
            "emergency_readiness_score", "critical_care_score",
            "service_richness_score", "infrastructure_completeness_score",
            "referral_complexity_score", "healthcare_maturity_score",
            "capability_graph_nodes", "capability_graph_edges",
            "capability_dependency_gaps", "capability_graph_summary",
            "specialties_parsed", "procedure_parsed", "equipment_parsed", "capability_parsed",
        },
    },
    "gold_regional_summary": {
        "key": "region_normalised",
        "region_col": "region_normalised",
        "cols": {
            "region_normalised", "total_facilities", "clinical_facility_count",
            "hospital_count", "clinical_hospital_count", "clinic_count",
            "public_facilities", "private_facilities", "ngo_count",
            "faith_based_count", "regional_faith_based_count", "government_facilities",
            "teaching_hospital_count", "referral_center_count", "specialist_hospital_count",
            "avg_doctors", "total_doctors", "avg_bed_capacity", "total_beds",
            "avg_completeness", "avg_geo_quality", "avg_clinical_complexity",
            "avg_evidence_weight", "avg_ghost_probability",
            "emergency_medicine_facilities", "obstetrics_facilities", "surgery_facilities",
            "pediatrics_facilities", "icu_facilities", "infectious_disease_facilities",
            "radiology_facilities", "mental_health_facilities",
            "facilities_with_procedures", "facilities_with_equipment",
            "facilities_with_capabilities", "volunteer_facilities",
            "region_centroid_lat", "region_centroid_lon", "total_region_anomalies",
            "avg_quality_risk", "avg_facility_desert_score",
            "avg_emergency_readiness", "avg_critical_care_score",
            "avg_service_richness_score", "avg_infrastructure_completeness_score",
            "avg_referral_complexity_score", "avg_healthcare_maturity_score",
            "rag_ready_count", "rag_ready_rate", "clinical_ready_count",
            "region_population",
            "facilities_per_100k", "hospitals_per_100k", "beds_per_100k",
            "doctors_per_100k", "icu_facilities_per_100k",
            "surgery_facilities_per_100k", "maternity_facilities_per_100k",
            "public_private_ratio",
            "maternity_gap_score", "emergency_gap_score", "icu_gap_score",
            "surgical_access_gap_score", "public_private_imbalance_score",
            "all_specialties", "missing_critical_specialties",
            "critical_specialty_gap_count", "recommended_actions",
            "medical_desert_score", "desert_label",
        },
    },
    "gold_medical_desert_scores": {
        "key": "region",
        "region_col": "region",
        "cols": {
            "region", "schema_version", "scored_at",
            "total_facilities", "hospital_count", "clinic_count", "ngo_count",
            "volunteer_facilities", "teaching_hospitals", "referral_centers",
            "public_facilities", "private_facilities",
            "total_doctors", "total_beds", "doctors_per_100k", "beds_per_100k",
            "facilities_per_100k", "hospitals_per_100k", "region_population",
            "emergency_medicine_facilities", "surgery_facilities", "obstetrics_facilities",
            "icu_facilities", "pediatrics_facilities", "infectious_disease_facilities",
            "radiology_facilities", "mental_health_facilities",
            "critical_specialty_gap_count", "missing_critical_specialties", "all_specialties",
            "emergency_gap_score", "icu_gap_score", "surgical_access_gap_score",
            "maternity_gap_score",
            "avg_completeness", "avg_geo_quality", "avg_ghost_probability",
            "avg_quality_risk", "total_region_anomalies",
            "rag_ready_count", "rag_ready_rate",
            "avg_emergency_readiness", "avg_critical_care_score",
            "avg_service_richness_score", "avg_infrastructure_completeness_score",
            "avg_referral_complexity_score", "avg_healthcare_maturity_score",
            "density_component", "specialty_component",
            "integrity_component", "confidence_component",
            "summary_mds", "blended_mds", "medical_desert_score",
            "desert_label", "mds_label",
            "score_confidence", "score_rationale",
            "centroid_lat", "centroid_lon",
            "recommended_actions", "method_version",
        },
    },
    "gold_anomaly_flags": {
        "key": "unique_id",
        "region_col": "region_normalised",
        "anomaly_count_col": "total_anomaly_flags",
        "cols": {
            "unique_id", "name", "city_clean", "region_normalised",
            "facility_type_clean", "facility_tier_label",
            "service_maturity_label", "service_maturity_tier",
            "organization_type_clean", "operatorTypeId", "source_trust",
            "latitude", "longitude", "geo_quality_score",
            "number_doctors_int", "capacity_int",
            "data_completeness_score", "evidence_absence_confidence",
            "capability_confidence", "capability_is_valid",
            "ghost_probability_score", "ghost_review_priority",
            "quality_risk_score", "clinical_risk_score",
            "operational_risk_score", "integrity_risk_score",
            "emergency_readiness_score", "critical_care_score", "rag_quality_score",
            "is_planning_ready", "is_search_ready", "is_clinical_ready",
            "has_emergency_medicine", "has_surgery", "has_icu", "has_obstetrics",
            "has_radiology", "has_infectious_disease", "has_mental_health", "has_pediatrics",
            "is_hospital", "is_clinic", "is_ngo", "is_faith_based",
            "is_teaching_hospital", "is_referral_center",
            "procedure_count", "equipment_count", "capability_count", "specialty_count",
            "total_anomaly_flags",
            "composite_anomaly_score", "anomaly_risk_level",
            "continuity_risk_score", "continuity_risk_flags",
            "high_continuity_risk", "identity_duplicate_flag",
            "identity_duplicate_risk", "identity_duplicate_cluster",
            "peer_capability_zscore", "peer_procedure_zscore",
            "peer_outlier_high_cap", "peer_outlier_low_equip",
            "quality_flag_taxonomy",
            "stat_anomaly_capability_inflation", "stat_anomaly_hospital_no_doctors",
            "stat_anomaly_clinic_claims_icu", "stat_anomaly_ghost_facility",
            "stat_anomaly_specialty_mismatch", "stat_anomaly_procedure_breadth",
            "enhanced_type_capability_mismatch", "enhanced_ghost_hospital",
            "enhanced_procedures_no_equipment", "enhanced_low_idp_confidence",
            "enhanced_suspicious_completeness", "enhanced_icu_no_infrastructure",
            "enhanced_implausible_doctor_bed_ratio", "enhanced_em_without_surgical_support",
            "enhanced_geo_contradiction", "enhanced_planning_overconfidence",
            "enhanced_graph_dependency_gap", "enhanced_peer_capability_outlier",
            "llm_priority_action", "llm_data_quality_score",
            "llm_confirmed_anomaly_count", "llm_anomaly_severity",
            "llm_clinical_assessment", "llm_false_positive_reason",
            "llm_recommended_quality_category",
            "specialties_enriched", "procedure_enriched",
            "equipment_enriched", "capability_enriched", "capability_anomalies",
            "medical_desert_score", "desert_label",
            "capability_dependency_gaps", "capability_graph_summary",
            "service_richness_score", "infrastructure_completeness_score",
            "referral_complexity_score", "healthcare_maturity_score",
        },
    },
    "gold_anomaly_report": {
        "key": "region",
        "region_col": "region",
        "cols": {
            "region", "total_facilities", "flagged_facilities", "flag_rate",
            "critical_risk", "high_risk", "medium_risk", "low_risk", "clean_facilities",
            "llm_processed", "llm_confirmed_count",
            "avg_data_quality", "avg_composite_score", "avg_continuity_risk",
            "high_continuity_risk_count", "identity_duplicate_risk_count",
            "avg_completeness", "avg_absence_confidence",
            "maturity_distribution", "top_anomaly_types", "worst_facilities",
        },
    },
    "gold_regional_priority": {
        "key": "region_normalised",
        "region_col": "region_normalised",
        "cols": {
            "region_normalised", "facility_count",
            "avg_desert_score", "avg_emergency_gap",
            "avg_continuity_fragility", "avg_anomaly_density",
            "avg_low_staff_density", "avg_low_equipment_density",
            "critical_facility_count", "high_risk_facility_count",
            "high_continuity_risk_count",
            "avg_emergency_readiness", "avg_data_completeness",
            "regional_priority_score", "priority_tier",
            "recommended_interventions",
        },
    },
}

# COMMAND ----------
# MAGIC %md ## 5. SQL System Prompt (Fully-Qualified Table Names)

# COMMAND ----------

SQL_GEN_SYSTEM_PROMPT = """You are a Spark SQL expert for Ghana healthcare intelligence.
Output ONLY valid PySpark SQL starting directly with SELECT. No markdown, no explanations.

=== MANDATORY: USE FULLY QUALIFIED TABLE NAMES (catalog.schema.table) ===

TABLE virtue_foundation.ghana.gold_idp_enriched  (~900 rows)
  Key: unique_id  |  Region: region_normalised
  Anomaly count: total_stat_anomalies
  Desert: desert_label, medical_desert_score
  Boolean: is_hospital, is_clinic, is_ngo, is_public, is_private, is_government,
    is_faith_based, is_teaching_hospital, is_referral_center,
    has_emergency_medicine, has_obstetrics, has_surgery, has_pediatrics,
    has_icu, has_radiology, has_infectious_disease, has_mental_health,
    accepts_volunteers_bool, ngo_serves_ghana, capability_is_valid
  Numeric (COALESCE all): number_doctors_int, capacity_int,
    procedure_count, equipment_count, capability_count, specialty_count,
    data_completeness_score, medical_desert_score, ghost_probability_score,
    service_richness_score, infrastructure_completeness_score,
    healthcare_maturity_score, referral_complexity_score,
    emergency_readiness_score, critical_care_score
  JSON STRING cols (NEVER GROUP BY): procedure_enriched, equipment_enriched,
    capability_enriched, specialties_enriched, capability_graph_summary,
    capability_dependency_gaps
  Native ARRAY cols (use array_contains()):
    specialties_parsed, procedure_parsed, equipment_parsed, capability_parsed
  camelCase cols: facilityTypeId, operatorTypeId  ← exact case required

TABLE virtue_foundation.ghana.gold_facilities_enriched  (~900 rows)
  Same structure as gold_idp_enriched BUT no capability graph or IDP enriched cols
  Anomaly count: total_stat_anomalies

TABLE virtue_foundation.ghana.gold_regional_summary  (17 rows)
  Key: region_normalised
  Cols: total_facilities, hospital_count, clinic_count, ngo_count,
    public_facilities, private_facilities, teaching_hospital_count,
    avg_doctors, total_doctors, avg_bed_capacity, total_beds,
    icu_facilities, emergency_medicine_facilities, obstetrics_facilities,
    surgery_facilities, pediatrics_facilities,
    facilities_per_100k, hospitals_per_100k, beds_per_100k, doctors_per_100k,
    maternity_gap_score, emergency_gap_score, icu_gap_score,
    surgical_access_gap_score, critical_specialty_gap_count,
    medical_desert_score, desert_label,
    avg_ghost_probability, avg_quality_risk, total_region_anomalies,
    region_centroid_lat, region_centroid_lon

TABLE virtue_foundation.ghana.gold_medical_desert_scores  (17 rows)
  Key: region  ← NOT region_normalised
  BOTH desert_label AND mds_label exist (same values)
  Cols: centroid_lat, centroid_lon  ← NOT region_centroid_*
  Numeric: medical_desert_score, density_component, specialty_component,
    integrity_component, confidence_component, score_confidence
  Specialty gaps: emergency_gap_score, icu_gap_score,
    surgical_access_gap_score, maternity_gap_score

TABLE virtue_foundation.ghana.gold_anomaly_flags  (~900 rows)
  Key: unique_id  |  Region: region_normalised
  Anomaly count: total_anomaly_flags  ← use THIS (NOT total_stat_anomalies)
  composite_anomaly_score, anomaly_risk_level IN ('CRITICAL','HIGH','MEDIUM','LOW','CLEAN')
  Stat flags: stat_anomaly_capability_inflation, stat_anomaly_hospital_no_doctors,
    stat_anomaly_clinic_claims_icu, stat_anomaly_ghost_facility,
    stat_anomaly_specialty_mismatch, stat_anomaly_procedure_breadth
  Enhanced flags: enhanced_type_capability_mismatch, enhanced_ghost_hospital,
    enhanced_procedures_no_equipment, enhanced_icu_no_infrastructure,
    enhanced_implausible_doctor_bed_ratio, enhanced_em_without_surgical_support,
    enhanced_peer_capability_outlier

TABLE virtue_foundation.ghana.gold_anomaly_report  (17 rows)
  Key: region  ← NOT region_normalised
  Cols: critical_risk, high_risk, medium_risk, low_risk, clean_facilities,
    flag_rate, avg_data_quality, avg_composite_score, avg_continuity_risk,
    worst_facilities

TABLE virtue_foundation.ghana.gold_regional_priority  (17 rows)
  Key: region_normalised
  Cols: regional_priority_score, priority_tier IN ('P1','P2','P3','P4'),
    avg_desert_score, avg_emergency_gap, avg_continuity_fragility,
    critical_facility_count, high_risk_facility_count,
    recommended_interventions

=== SQL RULES (MANDATORY) ===
1. ALWAYS use full table path: virtue_foundation.ghana.gold_* (NEVER bare table name)
2. Boolean: WHERE is_hospital = true   (NOT '1', NOT = 1)
3. ARRAY: array_contains(specialties_parsed, 'emergencyMedicine')
4. JSON string (NEVER GROUP BY on these): procedure_enriched, capability_enriched, etc.
5. OR conditions: wrap in parens: WHERE is_hospital = true AND (cond1 OR cond2)
6. COALESCE all nullable numerics: COALESCE(number_doctors_int, 0)
7. LIMIT 50 always
8. Exact camelCase: facilityTypeId, operatorTypeId
9. gold_medical_desert_scores key is 'region' not 'region_normalised'
10. gold_anomaly_report key is 'region' not 'region_normalised'

SPECIALTY NAMES (camelCase, exact):
  emergencyMedicine, gynecologyAndObstetrics, generalSurgery, internalMedicine,
  familyMedicine, pediatrics, cardiology, orthopedicSurgery, dentistry,
  ophthalmology, radiology, infectiousDiseases (with 's'), criticalCareMedicine,
  psychiatry, anesthesia, nephrology, medicalOncology
"""

# COMMAND ----------
# MAGIC %md ## 6. SQL Utilities (Table-Name Qualification Fix)

# COMMAND ----------

_SPEC_NORM: Dict[str, str] = {
    "'infectiousDisease'":    "'infectiousDiseases'",
    '"infectiousDisease"':    '"infectiousDiseases"',
    "'gynecologyObstetrics'": "'gynecologyAndObstetrics'",
    "'gynecology_obstetrics'":"'gynecologyAndObstetrics'",
    "'criticalcare'":         "'criticalCareMedicine'",
    "'critical_care'":        "'criticalCareMedicine'",
    "'generalsurgery'":       "'generalSurgery'",
    "'emergencymedicine'":    "'emergencyMedicine'",
}

_JSON_STR_COLS = frozenset([
    "procedure_enriched", "equipment_enriched", "capability_enriched",
    "specialties_enriched", "capability_graph_summary", "capability_dependency_gaps",
    "capability_graph_nodes", "capability_graph_edges", "recommended_actions",
    "capability_anomalies", "maturity_distribution", "top_anomaly_types",
    "worst_facilities", "recommended_interventions",
])

_COL_ALIASES: Dict[str, str] = {
    "operatortypeid":   "operatorTypeId",
    "facilitytypeid":   "facilityTypeId",
    "mds_score":        "medical_desert_score",
    "region_label":     "desert_label",
}


def _clean_sql(raw: str) -> str:
    if not raw:
        return ""
    clean = re.sub(r"^```(?:sql|python)?\s*", "", raw.strip(), flags=re.IGNORECASE)
    clean = re.sub(r"\s*```\s*$", "", clean).strip()
    clean = re.sub(r"--[^\n]*", " ", clean)
    clean = re.sub(
        r"^(?:python|sql|Here\s+is.*?:|Note:|Output:)[^\n]*\n",
        "", clean, flags=re.IGNORECASE | re.MULTILINE,
    )
    m = re.search(r"(SELECT\b.+?)(?=\n\n|\Z)", clean, re.IGNORECASE | re.DOTALL)
    if m:
        return m.group(1).strip()
    m2 = re.search(r"(SELECT\b.*)", clean, re.IGNORECASE | re.DOTALL)
    if m2:
        return m2.group(1).strip()
    return clean.strip()


def _qualify_table_names(sql: str) -> str:
    """
    FIX #1: Replace bare gold_* table names with fully-qualified catalog.schema.table.
    Prevents TABLE_OR_VIEW_NOT_FOUND when the LLM omits the schema prefix.
    """
    for short, full in _SHORT_TO_FULL.items():
        # Only replace occurrences NOT already preceded by 'ghana.'
        sql = re.sub(
            rf'(?<![.\w]){re.escape(short)}(?![.\w])',
            full,
            sql,
            flags=re.IGNORECASE,
        )
    # Remove any accidental double-prefixing
    for short, full in _SHORT_TO_FULL.items():
        double = f"{full}.{full}"
        if double in sql:
            sql = sql.replace(double, full)
        # Also fix virtue_foundation.ghana.virtue_foundation.ghana.X pattern
        double2 = f"virtue_foundation.ghana.{full}"
        if double2 in sql:
            sql = sql.replace(double2, full)
    return sql


def _post_process_sql(sql: str) -> str:
    for wrong, right in _SPEC_NORM.items():
        sql = sql.replace(wrong, right)
    for wrong, right in _COL_ALIASES.items():
        sql = re.sub(rf"\b{wrong}\b", right, sql, flags=re.IGNORECASE)
    sql = re.sub(r"\boperatortypeid\b", "operatorTypeId", sql, flags=re.IGNORECASE)
    sql = re.sub(r"\bfacilitytypeid\b",  "facilityTypeId",  sql, flags=re.IGNORECASE)
    sql = re.sub(
        r"\bAND\s+(array_contains\([^)]+\))\s+OR\s+([\w_]+\s+LIKE\s+'[^']+')",
        r"AND (\1 OR \2)",
        sql, flags=re.IGNORECASE,
    )
    # Qualify table names LAST (after other fixes)
    sql = _qualify_table_names(sql)
    return sql


def _guard_json_group_by(sql: str) -> Optional[str]:
    for col in _JSON_STR_COLS:
        if re.search(rf"\bGROUP\s+BY\s+{col}\b", sql, re.IGNORECASE):
            return f"""
SELECT
    region_normalised,
    COUNT(*) AS total_facilities,
    AVG(COALESCE(procedure_count, 0)) AS avg_procedure_count,
    SUM(CASE WHEN procedure_count <= 2 THEN 1 ELSE 0 END) AS few_procedures,
    SUM(CASE WHEN procedure_count >= 8 THEN 1 ELSE 0 END) AS many_procedures,
    AVG(COALESCE(equipment_count, 0)) AS avg_equipment_count,
    AVG(data_completeness_score) AS avg_completeness
FROM {cfg.IDP_TABLE}
WHERE is_hospital = true OR is_clinic = true
GROUP BY region_normalised
ORDER BY avg_procedure_count DESC
LIMIT 50
""".strip()
    return None

# COMMAND ----------
# MAGIC %md ## 7. Capability Graph Engine

# COMMAND ----------

class CapabilityGraphEngine:
    def analyse_facility(self, row: pd.Series) -> Dict:
        findings: List[Dict] = []
        risk_level = "LOW"

        graph_summary = safe_str(row.get("capability_graph_summary", ""))
        dep_gaps_raw  = safe_str(row.get("capability_dependency_gaps", ""))
        svc_richness  = safe_float(row.get("service_richness_score", 0.0))
        infra_score   = safe_float(row.get("infrastructure_completeness_score", 0.0))
        maturity      = safe_float(row.get("healthcare_maturity_score", 0.0))
        referral_cmplx= safe_float(row.get("referral_complexity_score", 0.0))

        dep_gaps: List[str] = []
        if dep_gaps_raw:
            try:
                parsed = json.loads(dep_gaps_raw)
                dep_gaps = parsed if isinstance(parsed, list) else [str(parsed)]
            except Exception:
                dep_gaps = [dep_gaps_raw[:200]]

        has_icu   = bool(row.get("has_icu", False))
        has_surg  = bool(row.get("has_surgery", False))
        equip_cnt = safe_int(row.get("equipment_count", 0))
        proc_cnt  = safe_int(row.get("procedure_count", 0))

        if has_icu and equip_cnt == 0 and infra_score < 0.2:
            findings.append({"type": "ICU_NO_INFRASTRUCTURE", "severity": "HIGH",
                              "description": "ICU claimed with zero equipment evidence",
                              "confidence": 0.90})
            risk_level = "HIGH"

        if has_surg and equip_cnt == 0 and proc_cnt < 2:
            findings.append({"type": "SURGERY_NO_SUPPORT", "severity": "HIGH",
                              "description": "Surgery claimed but equipment=0 and procedures<2",
                              "confidence": 0.88})
            risk_level = "HIGH"

        if svc_richness > 0.6 and equip_cnt == 0:
            findings.append({"type": "RICHNESS_EQUIPMENT_MISMATCH", "severity": "MEDIUM",
                              "description": f"service_richness={svc_richness:.2f} but equipment=0",
                              "confidence": 0.82})
            if risk_level == "LOW":
                risk_level = "MEDIUM"

        if maturity > 0.5 and infra_score < 0.15:
            findings.append({"type": "MATURITY_INFRA_MISMATCH", "severity": "MEDIUM",
                              "description": f"maturity={maturity:.2f} but infra={infra_score:.2f}",
                              "confidence": 0.80})
            if risk_level == "LOW":
                risk_level = "MEDIUM"

        if dep_gaps:
            findings.append({"type": "CAPABILITY_DEPENDENCY_GAPS",
                              "severity": "HIGH" if len(dep_gaps) > 3 else "MEDIUM",
                              "description": f"{len(dep_gaps)} gaps: {', '.join(dep_gaps[:4])}",
                              "confidence": 0.87})
            if len(dep_gaps) > 3 and risk_level != "HIGH":
                risk_level = "HIGH"

        if graph_summary and len(graph_summary) > 30:
            findings.append({"type": "GRAPH_SUMMARY", "severity": "INFO",
                              "description": graph_summary[:300], "confidence": 0.95})

        return {
            "facility_id":                  safe_str(row.get("unique_id", "")),
            "facility_name":                safe_str(row.get("name", "")),
            "risk_level":                   risk_level,
            "findings":                     findings,
            "service_richness_score":       round(svc_richness, 3),
            "infrastructure_completeness":  round(infra_score, 3),
            "healthcare_maturity_score":    round(maturity, 3),
            "referral_complexity_score":    round(referral_cmplx, 3),
            "dependency_gap_count":         len(dep_gaps),
            "graph_summary_excerpt":        graph_summary[:200],
            "overall_confidence":           0.85,
        }

    def batch_analyse(self, df: pd.DataFrame, top_n: int = 20) -> List[Dict]:
        if "total_stat_anomalies" in df.columns:
            df = df.sort_values("total_stat_anomalies", ascending=False)
        results = []
        for _, row in df.head(top_n).iterrows():
            r = self.analyse_facility(row)
            if r["findings"]:
                results.append(r)
        return results


cap_graph_engine = CapabilityGraphEngine()
print("CapabilityGraphEngine ready")

# COMMAND ----------
# MAGIC %md ## 8. Confidence Engine

# COMMAND ----------

class ConfidenceEngine:
    def compute(self, state: AgentStateV2) -> Tuple[float, float, float]:
        signals: List[float] = []
        halluc_risks: List[float] = []

        if state.get("sql_results"):
            try:
                rows = json.loads(state["sql_results"])
                if isinstance(rows, list) and len(rows) > 0 and "error" not in rows[0]:
                    signals.append(min(1.0, len(rows) / 10.0))
                    halluc_risks.append(0.1)
                else:
                    signals.append(0.2)
                    halluc_risks.append(0.5)
            except Exception:
                signals.append(0.4)
                halluc_risks.append(0.3)

        rag = state.get("rag_results") or []
        if rag:
            avg_score = sum(r.get("score", 0) for r in rag) / len(rag)
            signals.append(min(1.0, avg_score * 1.5))
            cited = sum(1 for r in rag if r.get("citations"))
            halluc_risks.append(max(0.1, 0.4 - cited * 0.05))
        else:
            halluc_risks.append(0.5)

        if state.get("anomaly_results"):
            signals.append(0.85)
            halluc_risks.append(0.15)
        if state.get("desert_data"):
            signals.append(0.90)
            halluc_risks.append(0.10)
        if state.get("geo_results") and not (state["geo_results"] or {}).get("error"):
            signals.append(0.80)
            halluc_risks.append(0.12)

        if not signals:
            return 0.5, 0.6, 0.4

        confidence = sum(signals) / len(signals)
        halluc     = sum(halluc_risks) / len(halluc_risks)
        retrieval  = min(1.0, len(signals) / 3.0)
        return round(confidence, 3), round(halluc, 3), round(retrieval, 3)


confidence_engine = ConfidenceEngine()

# COMMAND ----------
# MAGIC %md ## 9. Query Planner

# COMMAND ----------

QUERY_PLAN_PROMPT = """You are a healthcare query planner for Ghana.
Classify the query and return a JSON execution plan.

QUERY TYPES:
  facility_lookup, facility_comparison, regional_analysis, desert_analysis,
  anomaly_analysis, capability_validation, geo_search, specialty_gap_analysis,
  healthcare_planning, risk_assessment, infrastructure_analysis,
  ngo_coordination, capability_graph_reasoning, priority_analysis, general

AGENTS:
  sql, rag, geo, anomaly, desert, graph, priority, medical, planning, ngo, map

Return ONLY JSON:
{
  "query_type": "...",
  "complexity": "simple|moderate|complex",
  "primary_agent": "...",
  "execution_order": ["agent1", "agent2"],
  "required_tables": ["virtue_foundation.ghana.table1"],
  "confidence": 0.9,
  "extracted_params": {
    "region": null, "city": null, "specialty": null,
    "facility_type": null, "radius_km": null, "risk_level": null
  },
  "reasoning": "one sentence"
}"""


def plan_query(query: str) -> Dict:
    q = query.lower()

    HEURISTIC_PLANS: List[Tuple[List[str], Dict]] = [
        ([r"\bwithin\s*\d+\s*km\b", r"\bradius\b", r"\bnearby\b", r"\bcold\s+spot\b"],
         {"query_type": "geo_search", "primary_agent": "geo",
          "execution_order": ["geo", "medical"], "complexity": "moderate",
          "required_tables": [cfg.IDP_TABLE]}),
        ([r"suspicious", r"anomaly", r"flagged", r"inflated", r"ghost",
          r"unrealistic", r"shouldn.t\s+move", r"things.*move.*together",
          r"claim.*without", r"procedure.*breadth", r"enhanced.*flag"],
         {"query_type": "anomaly_analysis", "primary_agent": "anomaly",
          "execution_order": ["anomaly", "graph", "medical"], "complexity": "complex",
          "required_tables": [cfg.ANOMALY_FLAGS, cfg.ANOMALY_REPORT]}),
        ([r"medical\s+desert", r"underserved", r"specialty\s+gap",
          r"most\s+underserved", r"critical\s+desert", r"no\s+icu\s+region"],
         {"query_type": "desert_analysis", "primary_agent": "desert",
          "execution_order": ["desert", "priority", "medical"], "complexity": "moderate",
          "required_tables": [cfg.DESERT_TABLE, cfg.PRIORITY_TABLE]}),
        ([r"action\s+plan", r"intervention", r"deploy.*specialist",
          r"recommendation", r"resource\s+allocation", r"what\s+should.*do"],
         {"query_type": "healthcare_planning", "primary_agent": "planning",
          "execution_order": ["planning", "priority", "ngo"], "complexity": "complex",
          "required_tables": [cfg.PRIORITY_TABLE, cfg.REGIONAL_TABLE]}),
        ([r"priority\s+tier", r"most\s+urgent", r"intervention\s+priority",
          r"p1\s+region", r"highest.*priority"],
         {"query_type": "priority_analysis", "primary_agent": "priority",
          "execution_order": ["priority", "desert", "medical"], "complexity": "moderate",
          "required_tables": [cfg.PRIORITY_TABLE, cfg.DESERT_TABLE]}),
        ([r"\bngo\b", r"international\s+org", r"ngos\s+in", r"overlap.*ngo",
          r"charity.*gap"],
         {"query_type": "ngo_coordination", "primary_agent": "ngo",
          "execution_order": ["ngo", "desert"], "complexity": "moderate",
          "required_tables": [cfg.IDP_TABLE, cfg.REGIONAL_TABLE]}),
        ([r"capability\s+graph", r"dependency\s+gap", r"infrastructure.*missing",
          r"service.*maturity", r"healthcare.*maturity"],
         {"query_type": "capability_graph_reasoning", "primary_agent": "graph",
          "execution_order": ["graph", "medical"], "complexity": "complex",
          "required_tables": [cfg.IDP_TABLE, cfg.ANOMALY_FLAGS]}),
        ([r"oversupply", r"scarcity.*complex", r"concentration.*procedure",
          r"procedure.*depend.*few", r"which.*procedure.*few\s+facilit"],
         {"query_type": "specialty_gap_analysis", "primary_agent": "sql",
          "execution_order": ["sql", "medical"], "complexity": "moderate",
          "required_tables": [cfg.IDP_TABLE]}),
        ([r"how\s+many", r"count\s+of", r"\btotal\b", r"number\s+of",
          r"percentage", r"list\s+all", r"rank.*region", r"which\s+region.*most",
          r"average", r"correlation", r"per\s+region", r"icu.*per"],
         {"query_type": "regional_analysis", "primary_agent": "sql",
          "execution_order": ["sql"], "complexity": "simple",
          "required_tables": [cfg.IDP_TABLE]}),
        ([r"services.*offer", r"what.*offer", r"korle\s+bu",
          r"teaching\s+hospital", r"find\s+facilit", r"dialysis",
          r"what.*provide", r"\bhiv\b", r"malaria.*treat"],
         {"query_type": "facility_lookup", "primary_agent": "rag",
          "execution_order": ["rag"], "complexity": "simple",
          "required_tables": [cfg.IDP_TABLE]}),
    ]

    for patterns, plan_base in HEURISTIC_PLANS:
        if any(re.search(p, q) for p in patterns):
            return {
                "confidence": 0.88,
                "extracted_params": {"region": None, "city": None, "specialty": None,
                                     "facility_type": None, "radius_km": None,
                                     "risk_level": None},
                **plan_base,
            }

    resp   = call_llama(
        [{"role": "user", "content": f"Query: {query}"}],
        system_prompt=QUERY_PLAN_PROMPT,
        max_tokens=350, temperature=0.0,
    )
    parsed = parse_json_llm(resp, fallback={})
    if not parsed.get("query_type"):
        parsed = {
            "query_type": "facility_lookup", "primary_agent": "rag",
            "execution_order": ["rag"], "complexity": "simple",
            "required_tables": [cfg.IDP_TABLE], "confidence": 0.5,
        }
    parsed.setdefault("extracted_params", {})
    for k in ("region", "city", "specialty", "facility_type", "radius_km", "risk_level"):
        parsed["extracted_params"].setdefault(k, None)
    return parsed

# COMMAND ----------
# MAGIC %md ## 10. Router Node

# COMMAND ----------

def router_node(state: AgentStateV2) -> AgentStateV2:
    t0   = time.time()
    plan = plan_query(state["query"])

    primary    = plan.get("primary_agent", "rag")
    exec_order = [a for a in plan.get("execution_order", []) if a != primary]

    payload = _node_payload(
        "router", NodeStatus.SUCCESS,
        f"type={plan['query_type']} primary={primary} chain={exec_order}",
        confidence=plan.get("confidence", 0.8),
        results=[plan], evidence=[],
        timing_ms=int((time.time() - t0) * 1000),
    )
    cite = _step_cite(
        "router", state["query"],
        f"{plan['query_type']} → {primary}+{exec_order}",
        ["heuristic+llm_planner"], plan.get("confidence", 0.8),
    )
    return {
        **state,
        "query_type":       plan["query_type"],
        "query_complexity": plan.get("complexity", "moderate"),
        "execution_plan":   plan,
        "sub_agents":       exec_order,
        "steps":            [f"Router → {plan['query_type']} | {primary} → {exec_order}"],
        "step_citations":   state.get("step_citations", []) + [cite],
        "node_payloads":    state.get("node_payloads", []) + [payload],
    }

# COMMAND ----------
# MAGIC %md ## 11. SQL Node (with table qualification)

# COMMAND ----------

def sql_node(state: AgentStateV2) -> AgentStateV2:
    t0 = time.time()

    with _mlflow_nested("sql_node") as _run:
        if _run:
            mlflow.set_tag("step",  "sql_node")
            mlflow.set_tag("query", state["query"][:200])

        sql_raw   = call_llama(
            [{"role": "user", "content": f"Question: {state['query']}\n\nSQL:"}],
            system_prompt=SQL_GEN_SYSTEM_PROMPT,
            max_tokens=700, temperature=0.0,
        )
        sql_clean = _clean_sql(sql_raw)
        sql_clean = _post_process_sql(sql_clean)   # FIX: qualifies table names
        json_fix  = _guard_json_group_by(sql_clean)
        if json_fix:
            sql_clean = json_fix
            if _run:
                mlflow.set_tag("json_groupby_fixed", "true")

        if _run:
            mlflow.log_text(sql_clean, "generated_sql.sql")

        result_json = ""
        row_count   = 0
        tables_used: List[str] = []
        error_msg   = ""
        status      = NodeStatus.SUCCESS

        try:
            result_pd   = spark.sql(sql_clean).limit(50).toPandas()
            result_json = result_pd.to_json(orient="records", indent=2, default_handler=str)
            row_count   = len(result_pd)
            tables_used = re.findall(r"FROM\s+([\w.]+)", sql_clean, re.IGNORECASE)
            if _run:
                mlflow.log_metric("rows_returned", row_count)
        except Exception as e:
            error_msg   = str(e)[:300]
            result_json = json.dumps({"error": error_msg, "sql_attempted": sql_clean[:400]})
            status      = NodeStatus.ERROR
            if _run:
                mlflow.set_tag("sql_error", error_msg[:200])

    ev = _evidence_bundle(
        source_table=tables_used[0] if tables_used else "unknown",
        row_ids=[], cols=[],
        confidence=0.85 if status == NodeStatus.SUCCESS else 0.0,
        summary=f"SQL returned {row_count} rows",
        raw=[{"sql": sql_clean, "error": error_msg}],
    )
    payload = _node_payload(
        "sql_query", status,
        f"SQL: {row_count} rows" + (f" | error={error_msg[:80]}" if error_msg else ""),
        confidence=0.85 if status == NodeStatus.SUCCESS else 0.2,
        results=[], evidence=[dict(ev)],
        timing_ms=int((time.time() - t0) * 1000),
    )
    remaining = [a for a in state.get("sub_agents", []) if a != "sql"]
    return {
        **state,
        "sql_results":   result_json,
        "sql_row_count": row_count,
        "sub_agents":    remaining,
        "steps":         [f"SQL: {row_count} rows | {tables_used}"],
        "evidence_bundles": state.get("evidence_bundles", []) + [ev],
        "step_citations": state.get("step_citations", []) + [
            _step_cite("sql_node", state["query"], result_json[:200], tables_used, 0.85)
        ],
        "node_payloads": state.get("node_payloads", []) + [payload],
        "errors": state.get("errors", []) + ([error_msg] if error_msg else []),
    }

# COMMAND ----------
# MAGIC %md ## 12. RAG Node

# COMMAND ----------

_rag_idx_cache:  Optional[Any]  = None
_rag_meta_cache: Optional[Dict] = None


def _load_rag(force: bool = False) -> Tuple[Any, Dict]:
    global _rag_idx_cache, _rag_meta_cache
    if _rag_idx_cache is not None and not force:
        return _rag_idx_cache, _rag_meta_cache
    for idx_p, meta_p in [
        (cfg.RAG_INDEX_PATH, cfg.RAG_META_PATH),
        (cfg.VOLUME_INDEX,   cfg.VOLUME_META),
    ]:
        try:
            idx  = faiss.read_index(idx_p)
            with open(meta_p, encoding="utf-8") as f:
                store = json.load(f)
            _rag_idx_cache  = idx
            _rag_meta_cache = store
            print(f"FAISS loaded: {idx.ntotal:,} vectors from {idx_p}")
            return idx, store
        except Exception:
            continue
    raise RuntimeError("FAISS index unavailable. Run notebook 05 first.")


def _embed(query: str) -> np.ndarray:
    headers = {"Authorization": f"Bearer {cfg.DATABRICKS_TOKEN}",
               "Content-Type":  "application/json"}
    r = requests.post(cfg.EMBED_ENDPOINT, headers=headers,
                      json={"input": [query[:2000]]}, timeout=30)
    r.raise_for_status()
    vec = np.array([r.json()["data"][0]["embedding"]], dtype=np.float32)
    faiss.normalize_L2(vec)
    return vec


def _extract_rag_filters(query: str) -> Dict:
    q = query.lower()
    region = None
    REGIONS = [
        "greater accra", "ashanti", "western north", "bono east", "north east",
        "upper east", "upper west", "western", "eastern", "central", "volta",
        "northern", "oti", "savannah", "ahafo", "brong-ahafo",
    ]
    for r in sorted(REGIONS, key=len, reverse=True):
        if r in q:
            region = r.title()
            break
    ftype = next((t for t in ["hospital", "clinic", "ngo", "pharmacy", "maternity"] if t in q), None)
    SPEC_MAP = {
        "dialysis": "nephrology", "cardiology": "cardiology",
        "surgery": "generalSurgery", "emergency": "emergencyMedicine",
        "obstetric": "gynecologyAndObstetrics", "maternity": "gynecologyAndObstetrics",
        "pediatric": "pediatrics", "child": "pediatrics",
        "icu": "criticalCareMedicine", "radiology": "radiology",
        "hiv": "infectiousDiseases", "malaria": "infectiousDiseases",
        "infectious": "infectiousDiseases", "mental": "psychiatry",
        "dental": "dentistry", "eye": "ophthalmology",
    }
    spec   = next((sp for kw, sp in SPEC_MAP.items() if kw in q), None)
    volunt = any(w in q for w in ["volunteer", "accepts volunteer"])
    return {"region": region, "facility_type": ftype, "specialty": spec, "volunteer": volunt}


def rag_node(state: AgentStateV2) -> AgentStateV2:
    t0      = time.time()
    results: List[Dict] = []
    status  = NodeStatus.SUCCESS
    error   = ""

    with _mlflow_nested("rag_node") as _run:
        if _run:
            mlflow.set_tag("step", "rag_node")
        try:
            idx, store = _load_rag()
            q_vec      = _embed(state["query"])
            filters    = _extract_rag_filters(state["query"])
            k          = min(80, idx.ntotal)
            dists, idxs = idx.search(q_vec, k)

            for dist, i in zip(dists[0], idxs[0]):
                if not (0 <= i < len(store.get("documents", []))):
                    continue
                meta = store["metadata"][i] if i < len(store.get("metadata", [])) else {}

                if filters["region"] and meta.get("region", "").lower() != filters["region"].lower():
                    continue
                if filters["facility_type"] and \
                   filters["facility_type"] not in meta.get("facility_type", "").lower():
                    continue
                if filters["specialty"]:
                    specs = [s.lower() for s in meta.get("specialties", [])]
                    doc   = store["documents"][i].lower()
                    sp_lc = filters["specialty"].lower()
                    if sp_lc not in specs and sp_lc.replace("and","").replace(" ","") not in doc:
                        continue
                if filters["volunteer"] and not meta.get("accepts_volunteers", False):
                    continue

                cits = (
                    [c for c in meta.get("idp_citations", []) if len(str(c)) > 15]
                    or [c for c in meta.get("citations", []) if len(str(c)) > 10]
                )
                results.append({
                    "score":        float(dist),
                    "document":     store["documents"][i],
                    "metadata":     meta,
                    "citations":    cits,
                    "graph_summary":meta.get("capability_graph_summary", ""),
                    "dep_gaps":     meta.get("capability_dependency_gaps", ""),
                    "svc_richness": meta.get("service_richness_score", 0.0),
                    "maturity":     meta.get("healthcare_maturity_score", 0.0),
                })
                if len(results) >= 10:
                    break
            if _run:
                mlflow.log_metric("rag_docs", len(results))
        except RuntimeError as e:
            status = NodeStatus.FALLBACK
            error  = str(e)
        except Exception as e:
            status = NodeStatus.ERROR
            error  = str(e)[:200]

    payload = _node_payload(
        "rag_search", status,
        f"RAG: {len(results)} docs" + (f" | {error[:60]}" if error else ""),
        confidence=0.80 if results else 0.1,
        results=[], evidence=[],
        timing_ms=int((time.time() - t0) * 1000),
    )
    remaining = [a for a in state.get("sub_agents", []) if a != "rag"]
    return {
        **state,
        "rag_results":  results,
        "sub_agents":   remaining,
        "steps":        [f"RAG: {len(results)} results"],
        "step_citations": state.get("step_citations", []) + [
            _step_cite("rag_node", state["query"], f"{len(results)} docs",
                       [r["metadata"].get("unique_id", "") for r in results[:4]], 0.80)
        ],
        "node_payloads": state.get("node_payloads", []) + [payload],
        "warnings": state.get("warnings", []) + ([error] if error else []),
    }

# COMMAND ----------
# MAGIC %md ## 13. Geo Node

# COMMAND ----------

def geo_node(state: AgentStateV2) -> AgentStateV2:
    t0     = time.time()
    status = NodeStatus.SUCCESS
    q      = state["query"].lower()

    try:
        rm = re.search(r"(\d+)\s*km", q)
        radius_km = float(rm.group(1)) if rm else 50.0

        center_name = "accra"
        center_lat, center_lon = 5.5600, -0.1969
        for city in sorted(cfg.GHANA_CITY_COORDS.keys(), key=len, reverse=True):
            if city in q:
                center_name = city
                center_lat, center_lon = cfg.GHANA_CITY_COORDS[city]
                break

        src   = cfg.IDP_TABLE if table_exists(cfg.IDP_TABLE) else cfg.FACILITIES_TABLE
        avail = get_table_cols(src)
        geo_cols = [c for c in [
            "unique_id", "name", "facility_type_clean", "region_normalised",
            "city_clean", "latitude", "longitude",
            "has_emergency_medicine", "has_surgery", "has_icu", "has_obstetrics",
            "has_radiology", "has_infectious_disease", "has_pediatrics",
            "data_completeness_score", "medical_desert_score", "desert_label",
            "service_richness_score", "healthcare_maturity_score",
            "ghost_probability_score", "total_stat_anomalies",
        ] if c in avail]

        fac_pd = spark.table(src).select(*geo_cols).toPandas()
        fac_pd["latitude"]  = pd.to_numeric(fac_pd["latitude"],  errors="coerce")
        fac_pd["longitude"] = pd.to_numeric(fac_pd["longitude"], errors="coerce")
        fac_pd = fac_pd.dropna(subset=["latitude", "longitude"])
        GHANA_CENTER = (7.9465, -1.0232)
        fac_pd = fac_pd[~(
            (abs(fac_pd["latitude"] - GHANA_CENTER[0]) < 0.002) &
            (abs(fac_pd["longitude"] - GHANA_CENTER[1]) < 0.002)
        )]

        fac_pd["distance_km"] = fac_pd.apply(
            lambda r: haversine_km(center_lat, center_lon,
                                    float(r["latitude"]), float(r["longitude"])), axis=1,
        )
        within_r = fac_pd[fac_pd["distance_km"] <= radius_km].sort_values("distance_km")

        SPEC_FILTERS = {
            "malaria": "has_infectious_disease", "hiv": "has_infectious_disease",
            "surgery": "has_surgery",
            "emergency": "has_emergency_medicine", "icu": "has_icu",
            "obstetric": "has_obstetrics", "maternity": "has_obstetrics",
            "radiology": "has_radiology", "pediatric": "has_pediatrics",
        }
        spec_col = next((col for kw, col in SPEC_FILTERS.items() if kw in q), None)
        filtered = within_r[within_r[spec_col] == True] if (spec_col and spec_col in within_r.columns) else within_r

        cold_spots: List[Dict] = []
        cold_targets = {
            "no surgery": "has_surgery", "no emergency": "has_emergency_medicine",
            "no icu": "has_icu", "cold spot": "has_emergency_medicine",
        }
        for phrase, tgt_col in cold_targets.items():
            if phrase in q and tgt_col in fac_pd.columns:
                reg_cov = fac_pd.groupby("region_normalised")[tgt_col].any().reset_index()
                gap_rg  = reg_cov[~reg_cov[tgt_col]]["region_normalised"].tolist()
                for rg in gap_rg:
                    rf = fac_pd[fac_pd["region_normalised"] == rg]
                    cold_spots.append({
                        "region": rg, "missing_capability": tgt_col.replace("has_", ""),
                        "facility_count": len(rf),
                        "centroid_lat": round(float(rf["latitude"].mean()), 4) if len(rf) else 0,
                        "centroid_lon": round(float(rf["longitude"].mean()), 4) if len(rf) else 0,
                    })
                break

        facilities_out = []
        for _, r in filtered.head(20).iterrows():
            d = {k: (None if (isinstance(v, float) and v != v) else v)
                 for k, v in r.to_dict().items()}
            d["distance_km"] = round(float(d.get("distance_km", 0)), 2)
            facilities_out.append(d)

        geo_out = {
            "center":            {"name": center_name, "lat": center_lat, "lon": center_lon},
            "radius_km":         radius_km,
            "total_in_radius":   len(within_r),
            "specialty_matches": len(filtered),
            "specialty_filter":  spec_col,
            "facilities":        facilities_out[:15],
            "cold_spots":        cold_spots,
        }
        step_msg = (f"Geo: {len(filtered)} facilities within {radius_km}km "
                    f"of {center_name.title()} | {len(cold_spots)} cold spots")
    except Exception as e:
        geo_out  = {"error": str(e)}
        step_msg = f"Geo failed: {str(e)[:120]}"
        status   = NodeStatus.ERROR

    payload   = _node_payload("geo_calc", status, step_msg, 0.88, [], [],
                               int((time.time() - t0) * 1000))
    remaining = [a for a in state.get("sub_agents", []) if a != "geo"]
    return {
        **state,
        "geo_results": geo_out,
        "sub_agents":  remaining,
        "steps":       [step_msg],
        "step_citations": state.get("step_citations", []) + [
            _step_cite("geo_node", state["query"], step_msg, [src], 0.88)
        ],
        "node_payloads": state.get("node_payloads", []) + [payload],
    }

# COMMAND ----------
# MAGIC %md ## 14. Map Node

# COMMAND ----------

def map_node(state: AgentStateV2) -> AgentStateV2:
    t0 = time.time()
    try:
        src   = cfg.IDP_TABLE if table_exists(cfg.IDP_TABLE) else cfg.FACILITIES_TABLE
        avail = get_table_cols(src)
        cols  = [c for c in [
            "unique_id", "name", "facility_type_clean", "organization_type_clean",
            "city_clean", "region_normalised", "latitude", "longitude",
            "capacity_int", "number_doctors_int", "capability_is_valid",
            "accepts_volunteers_bool", "data_completeness_score",
            "total_stat_anomalies", "is_hospital", "is_ngo",
            "has_emergency_medicine", "has_surgery", "has_icu", "has_obstetrics",
            "medical_desert_score", "desert_label",
            "service_richness_score", "healthcare_maturity_score",
        ] if c in avail]

        fac_pd = spark.table(src).select(*cols).toPandas()

        try:
            d_pd = spark.table(cfg.DESERT_TABLE).select(
                "region", "medical_desert_score", "desert_label", "mds_label"
            ).toPandas()
            desert_map = dict(zip(d_pd["region"], d_pd["medical_desert_score"]))
            label_map  = dict(zip(d_pd["region"], d_pd["mds_label"]))
        except Exception:
            desert_map, label_map = {}, {}

        def _color(mds: float) -> str:
            if mds > 0.65: return "#dc2626"
            if mds > 0.50: return "#ea580c"
            if mds > 0.40: return "#ca8a04"
            return "#16a34a"

        features = []
        for _, row in fac_pd.iterrows():
            lat = safe_float(row.get("latitude"), 0.0)
            lon = safe_float(row.get("longitude"), 0.0)
            if lat == 0.0 and lon == 0.0:
                continue
            if abs(lat - 7.9465) < 0.002 and abs(lon + 1.0232) < 0.002:
                continue
            region = safe_str(row.get("region_normalised"), "Unknown")
            mds    = safe_float(desert_map.get(region) or row.get("medical_desert_score"), 0.5)
            label  = label_map.get(region) or safe_str(row.get("desert_label"), "Unknown")
            features.append({
                "type": "Feature",
                "geometry": {"type": "Point", "coordinates": [lon, lat]},
                "properties": {
                    "id":            safe_str(row.get("unique_id")),
                    "name":          safe_str(row.get("name")),
                    "type":          safe_str(row.get("facility_type_clean")).lower(),
                    "city":          safe_str(row.get("city_clean")),
                    "region":        region,
                    "capacity":      safe_int(row.get("capacity_int")),
                    "doctors":       safe_int(row.get("number_doctors_int")),
                    "desert_score":  round(mds, 4),
                    "desert_label":  label,
                    "has_anomaly":   not bool(row.get("capability_is_valid", True)),
                    "accepts_volunteers": bool(row.get("accepts_volunteers_bool")),
                    "completeness":  round(safe_float(row.get("data_completeness_score")), 3),
                    "has_emergency": bool(row.get("has_emergency_medicine")),
                    "has_surgery":   bool(row.get("has_surgery")),
                    "has_icu":       bool(row.get("has_icu")),
                    "svc_richness":  round(safe_float(row.get("service_richness_score")), 3),
                    "maturity":      round(safe_float(row.get("healthcare_maturity_score")), 3),
                    "color":         _color(mds),
                    "tooltip":       (f"{safe_str(row.get('name'))} | "
                                      f"{safe_str(row.get('facility_type_clean'))} | {region}"),
                },
            })

        geojson  = {"type": "FeatureCollection", "features": features}
        step_msg = f"Map: {len(features):,} markers"
    except Exception as e:
        geojson  = {"type": "FeatureCollection", "features": []}
        step_msg = f"Map failed: {e}"

    payload   = _node_payload("map_gen", NodeStatus.SUCCESS, step_msg, 0.95, [], [],
                               int((time.time() - t0) * 1000))
    remaining = [a for a in state.get("sub_agents", []) if a != "map"]
    return {
        **state, "map_data": geojson, "sub_agents": remaining, "steps": [step_msg],
        "node_payloads": state.get("node_payloads", []) + [payload],
    }

# COMMAND ----------
# MAGIC %md ## 15. Anomaly Node

# COMMAND ----------

def anomaly_node(state: AgentStateV2) -> AgentStateV2:
    t0      = time.time()
    status  = NodeStatus.SUCCESS
    anomalies: List[Dict] = []
    report:    List[Dict] = []

    with _mlflow_nested("anomaly_node") as _run:
        if _run:
            mlflow.set_tag("step", "anomaly_node")

        try:
            avail     = get_table_cols(cfg.ANOMALY_FLAGS)
            # FIX: intersect with actual schema to avoid column-not-found
            want_cols = [c for c in [
                "unique_id", "name", "city_clean", "region_normalised",
                "facility_type_clean", "facility_tier_label",
                "service_maturity_label", "service_maturity_tier",
                "total_anomaly_flags", "composite_anomaly_score", "anomaly_risk_level",
                "llm_anomaly_severity", "llm_priority_action",
                "llm_data_quality_score", "llm_confirmed_anomaly_count",
                "llm_clinical_assessment",
                "capability_anomalies",
                "continuity_risk_score", "high_continuity_risk",
                "peer_capability_zscore", "peer_procedure_zscore",
                "peer_outlier_high_cap", "peer_outlier_low_equip",
                "procedure_count", "equipment_count",
                "capability_count", "specialty_count",
                "data_completeness_score", "capability_confidence",
                "ghost_probability_score", "capability_is_valid",
                "medical_desert_score", "desert_label",
                "stat_anomaly_capability_inflation", "stat_anomaly_hospital_no_doctors",
                "stat_anomaly_ghost_facility", "stat_anomaly_procedure_breadth",
                "stat_anomaly_clinic_claims_icu", "stat_anomaly_specialty_mismatch",
                "enhanced_type_capability_mismatch", "enhanced_ghost_hospital",
                "enhanced_procedures_no_equipment", "enhanced_icu_no_infrastructure",
                "enhanced_implausible_doctor_bed_ratio",
                "enhanced_em_without_surgical_support",
                "enhanced_peer_capability_outlier",
                "enhanced_graph_dependency_gap",    # may not exist — guarded by avail
                "service_richness_score", "infrastructure_completeness_score",
                "healthcare_maturity_score", "referral_complexity_score",
                "capability_dependency_gaps", "capability_graph_summary",
            ] if c in avail]

            anomalies = (
                spark.table(cfg.ANOMALY_FLAGS)
                .select(*want_cols)
                .filter(F.col("anomaly_risk_level").isin("CRITICAL", "HIGH", "MEDIUM"))
                .orderBy(F.desc("total_anomaly_flags"), F.desc("composite_anomaly_score"))
                .limit(30)
                .toPandas()
                .infer_objects(copy=False)
                .fillna("")
                .to_dict(orient="records")
            )
            if _run:
                mlflow.log_metric("anomalies_returned", len(anomalies))
        except Exception as e:
            status = NodeStatus.FALLBACK
            if _run:
                mlflow.set_tag("anomaly_primary_fail", str(e)[:200])
            try:
                avail2   = get_table_cols(cfg.IDP_TABLE)
                fb_cols  = [c for c in [
                    "name", "city_clean", "region_normalised", "facility_type_clean",
                    "total_stat_anomalies", "capability_anomalies", "capability_confidence",
                    "stat_anomaly_capability_inflation", "stat_anomaly_hospital_no_doctors",
                    "stat_anomaly_procedure_breadth", "stat_anomaly_clinic_claims_icu",
                    "ghost_probability_score", "capability_is_valid",
                    "procedure_count", "equipment_count",
                    "service_richness_score", "infrastructure_completeness_score",
                ] if c in avail2]
                anomalies = (
                    spark.table(cfg.IDP_TABLE)
                    .select(*fb_cols)
                    .filter(F.col("total_stat_anomalies") > 0)
                    .orderBy(F.desc("total_stat_anomalies"))
                    .limit(30)
                    .toPandas()
                    .infer_objects(copy=False)
                    .fillna("")
                    .to_dict(orient="records")
                )
            except Exception:
                anomalies = []
                status    = NodeStatus.ERROR

        try:
            report = (
                spark.table(cfg.ANOMALY_REPORT)
                .orderBy(F.desc("flag_rate"))
                .limit(10)
                .toPandas()
                .infer_objects(copy=False)
                .fillna("")
                .to_dict(orient="records")
            )
        except Exception:
            pass

    step_msg = f"Anomaly: {len(anomalies)} flagged | report: {len(report)} regions"
    payload  = _node_payload(
        "anomaly_check", status, step_msg,
        confidence=0.88 if status == NodeStatus.SUCCESS else 0.55,
        results=anomalies[:5], evidence=[],
        timing_ms=int((time.time() - t0) * 1000),
    )
    remaining = [a for a in state.get("sub_agents", []) if a != "anomaly"]
    return {
        **state,
        "anomaly_results": anomalies,
        "sub_agents":      remaining,
        "steps":           [step_msg],
        "step_citations": state.get("step_citations", []) + [
            _step_cite("anomaly_node", state["query"], f"{len(anomalies)} anomalies",
                       [cfg.ANOMALY_FLAGS], 0.88)
        ],
        "node_payloads": state.get("node_payloads", []) + [payload],
    }

# COMMAND ----------
# MAGIC %md ## 16. Capability Graph Node

# COMMAND ----------

def graph_node(state: AgentStateV2) -> AgentStateV2:
    t0     = time.time()
    status = NodeStatus.SUCCESS

    with _mlflow_nested("graph_node") as _run:
        if _run:
            mlflow.set_tag("step", "graph_node")
        graph_results: List[Dict] = []
        try:
            avail = get_table_cols(cfg.IDP_TABLE)
            gcols = [c for c in [
                "unique_id", "name", "city_clean", "region_normalised",
                "facility_type_clean", "is_hospital", "is_clinic",
                "has_icu", "has_surgery", "has_emergency_medicine",
                "procedure_count", "equipment_count", "capability_count",
                "capability_graph_nodes", "capability_graph_edges",
                "capability_dependency_gaps", "capability_graph_summary",
                "service_richness_score", "infrastructure_completeness_score",
                "referral_complexity_score", "healthcare_maturity_score",
                "total_stat_anomalies", "ghost_probability_score",
                "data_completeness_score",
            ] if c in avail]

            filter_expr = F.lit(True)
            if "total_stat_anomalies" in avail:
                filter_expr = F.col("total_stat_anomalies") > 0

            idp_pd = (
                spark.table(cfg.IDP_TABLE)
                .select(*gcols)
                .filter(filter_expr)
                .orderBy(F.desc("total_stat_anomalies") if "total_stat_anomalies" in avail else F.lit(1))
                .limit(100)
                .toPandas()
                .fillna(0)
            )
            graph_results = cap_graph_engine.batch_analyse(idp_pd, top_n=30)

            # Supplement from anomaly flags
            try:
                af_avail = get_table_cols(cfg.ANOMALY_FLAGS)
                af_cols  = [c for c in [
                    "unique_id", "name", "region_normalised",
                    "enhanced_icu_no_infrastructure", "enhanced_procedures_no_equipment",
                    "enhanced_graph_dependency_gap", "enhanced_peer_capability_outlier",
                    "capability_dependency_gaps", "capability_graph_summary",
                    "service_richness_score", "healthcare_maturity_score",
                    "total_anomaly_flags",
                ] if c in af_avail]

                if len(af_cols) > 3:
                    af_pd = (
                        spark.table(cfg.ANOMALY_FLAGS)
                        .select(*af_cols)
                        .filter(F.col("total_anomaly_flags") >= 3
                                if "total_anomaly_flags" in af_avail else F.lit(True))
                        .limit(50)
                        .toPandas()
                        .fillna(0)
                    )
                    for rec in af_pd.to_dict(orient="records")[:10]:
                        triggered = [k for k in [
                            "enhanced_icu_no_infrastructure",
                            "enhanced_procedures_no_equipment",
                            "enhanced_graph_dependency_gap",
                        ] if rec.get(k)]
                        if triggered:
                            graph_results.append({
                                "facility_id":   safe_str(rec.get("unique_id")),
                                "facility_name": safe_str(rec.get("name")),
                                "risk_level":    "HIGH",
                                "source":        "anomaly_flags_enhanced",
                                "findings":      [
                                    {"type": k, "severity": "HIGH",
                                     "description": f"Enhanced flag: {k}", "confidence": 0.90}
                                    for k in triggered
                                ],
                                "healthcare_maturity_score": safe_float(rec.get("healthcare_maturity_score")),
                                "service_richness_score":    safe_float(rec.get("service_richness_score")),
                            })
            except Exception:
                pass

            if _run:
                mlflow.log_metric("graph_findings", len(graph_results))
        except Exception as e:
            status = NodeStatus.ERROR
            if _run:
                mlflow.set_tag("graph_error", str(e)[:200])

    step_msg  = f"Capability graph: {len(graph_results)} findings"
    payload   = _node_payload("capability_graph", status, step_msg,
                               0.87 if graph_results else 0.4,
                               graph_results[:5], [],
                               int((time.time() - t0) * 1000))
    remaining = [a for a in state.get("sub_agents", []) if a != "graph"]
    return {
        **state,
        "graph_results": graph_results,
        "sub_agents":    remaining,
        "steps":         [step_msg],
        "step_citations": state.get("step_citations", []) + [
            _step_cite("graph_node", state["query"], step_msg,
                       [cfg.IDP_TABLE, cfg.ANOMALY_FLAGS], 0.87)
        ],
        "node_payloads": state.get("node_payloads", []) + [payload],
    }

# COMMAND ----------
# MAGIC %md ## 17. Desert Node

# COMMAND ----------

def desert_node(state: AgentStateV2) -> AgentStateV2:
    t0 = time.time()

    with _mlflow_nested("desert_node") as _run:
        if _run:
            mlflow.set_tag("step", "desert_node")
        desert_data: Dict = {"all_regions": [], "most_underserved": [], "cold_spots": []}
        desert_top:  List[Dict] = []
        try:
            mds_avail = get_table_cols(cfg.DESERT_TABLE)
            mds_cols  = [c for c in [
                "region", "medical_desert_score", "desert_label", "mds_label",
                "score_confidence", "score_rationale",
                "density_component", "specialty_component",
                "integrity_component", "confidence_component",
                "emergency_gap_score", "icu_gap_score",
                "surgical_access_gap_score", "maternity_gap_score",
                "critical_specialty_gap_count", "missing_critical_specialties",
                "icu_facilities", "surgery_facilities", "obstetrics_facilities",
                "total_doctors", "total_beds", "total_facilities",
                "hospitals_per_100k", "beds_per_100k", "doctors_per_100k",
                "avg_emergency_readiness", "avg_critical_care_score",
                "avg_service_richness_score", "avg_healthcare_maturity_score",
                "centroid_lat", "centroid_lon", "recommended_actions",
            ] if c in mds_avail]

            mds_pd = (
                spark.table(cfg.DESERT_TABLE)
                .select(*mds_cols)
                .orderBy(F.desc("medical_desert_score"))
                .toPandas()
                .infer_objects(copy=False)
            )
            for col in ["missing_critical_specialties", "recommended_actions"]:
                if col in mds_pd.columns:
                    mds_pd[col] = mds_pd[col].apply(ensure_list)

            try:
                reg_avail = get_table_cols(cfg.REGIONAL_TABLE)
                reg_cols  = [c for c in [
                    "region_normalised", "ngo_count", "volunteer_facilities",
                    "avg_ghost_probability", "total_region_anomalies",
                    "missing_critical_specialties", "critical_specialty_gap_count",
                    "avg_emergency_readiness", "avg_healthcare_maturity_score",
                ] if c in reg_avail]
                reg_pd = spark.table(cfg.REGIONAL_TABLE).select(*reg_cols).toPandas().infer_objects(copy=False)
                for col in ["missing_critical_specialties"]:
                    if col in reg_pd.columns:
                        reg_pd[col] = reg_pd[col].apply(ensure_list)
                reg_dict = {r["region_normalised"]: r.to_dict() for _, r in reg_pd.iterrows()}
            except Exception:
                reg_dict = {}

            cold_spots = []
            for _, r in mds_pd.iterrows():
                miss = ensure_list(r.get("missing_critical_specialties", []))
                if len(miss) >= 3:
                    cold_spots.append({
                        "region":            r["region"],
                        "mds_label":         safe_str(r.get("mds_label") or r.get("desert_label")),
                        "mds_score":         round(safe_float(r["medical_desert_score"]), 4),
                        "score_confidence":  round(safe_float(r.get("score_confidence")), 3),
                        "missing_specialties": miss,
                        "emergency_gap":     round(safe_float(r.get("emergency_gap_score")), 3),
                        "icu_gap":           round(safe_float(r.get("icu_gap_score")), 3),
                        "centroid_lat":      safe_float(r.get("centroid_lat"), 7.94),
                        "centroid_lon":      safe_float(r.get("centroid_lon"), -1.02),
                        "recommended_actions": ensure_list(r.get("recommended_actions"))[:3],
                    })

            desert_top = []
            for rec in mds_pd.head(5).to_dict(orient="records"):
                clean = {k: (None if (isinstance(v, float) and v != v) else v)
                         for k, v in rec.items()}
                desert_top.append(clean)

            desert_data = {
                "all_regions":       mds_pd.to_dict(orient="records"),
                "most_underserved":  desert_top,
                "least_underserved": mds_pd.tail(3).to_dict(orient="records"),
                "cold_spots":        cold_spots,
                "regional_detail":   reg_dict,
                "severe_count":  int((mds_pd["mds_label"] == "Severe Desert").sum())
                                  if "mds_label" in mds_pd.columns else 0,
                "moderate_count": int((mds_pd["mds_label"] == "Moderate Desert").sum())
                                   if "mds_label" in mds_pd.columns else 0,
            }
            if _run:
                mlflow.log_metric("regions_loaded", len(mds_pd))
            step_msg = (f"Desert: {len(mds_pd)} regions | "
                        f"{desert_data['severe_count']} Severe | {len(cold_spots)} cold spots")
        except Exception as e:
            desert_data = {"error": str(e), "all_regions": []}
            desert_top  = []
            step_msg    = f"Desert failed: {e}"

    payload   = _node_payload("desert_check", NodeStatus.SUCCESS, step_msg, 0.90,
                               desert_top, [], int((time.time() - t0) * 1000))
    remaining = [a for a in state.get("sub_agents", []) if a != "desert"]
    return {
        **state,
        "desert_data":  desert_data,
        "desert_top":   desert_top,
        "sub_agents":   remaining,
        "steps":        [step_msg],
        "step_citations": state.get("step_citations", []) + [
            _step_cite("desert_node", state["query"], step_msg,
                       [cfg.DESERT_TABLE, cfg.REGIONAL_TABLE], 0.90)
        ],
        "node_payloads": state.get("node_payloads", []) + [payload],
    }

# COMMAND ----------
# MAGIC %md ## 18. Priority Node

# COMMAND ----------

def priority_node(state: AgentStateV2) -> AgentStateV2:
    t0 = time.time()

    with _mlflow_nested("priority_node") as _run:
        if _run:
            mlflow.set_tag("step", "priority_node")
        priority_data: Dict = {"all_regions": [], "p1_p2_regions": [], "total": 0}
        try:
            pr_avail = get_table_cols(cfg.PRIORITY_TABLE)
            pr_cols  = [c for c in [
                "region_normalised", "facility_count",
                "avg_desert_score", "avg_emergency_gap",
                "avg_continuity_fragility", "avg_anomaly_density",
                "avg_low_staff_density", "avg_low_equipment_density",
                "critical_facility_count", "high_risk_facility_count",
                "high_continuity_risk_count",
                "avg_emergency_readiness", "avg_data_completeness",
                "regional_priority_score", "priority_tier",
                "recommended_interventions",
            ] if c in pr_avail]

            pr_pd = (
                spark.table(cfg.PRIORITY_TABLE)
                .select(*pr_cols)
                .orderBy(F.desc("regional_priority_score"))
                .toPandas()
                .infer_objects(copy=False)
                .fillna("")
            )
            pr_pd["interventions_list"] = pr_pd["recommended_interventions"].apply(
                lambda v: [x.strip() for x in str(v).split("|") if x.strip()]
            )

            try:
                mds_pd = spark.table(cfg.DESERT_TABLE).select(
                    F.col("region").alias("region_normalised"),
                    F.col("mds_label"), F.col("medical_desert_score")
                ).toPandas()
                pr_pd = pr_pd.merge(mds_pd, on="region_normalised", how="left")
            except Exception:
                pass

            try:
                ar_pd = spark.table(cfg.ANOMALY_REPORT).select(
                    F.col("region").alias("region_normalised"),
                    F.col("flag_rate"), F.col("critical_risk"),
                    F.col("high_risk"), F.col("worst_facilities"),
                ).toPandas()
                pr_pd = pr_pd.merge(ar_pd, on="region_normalised", how="left")
            except Exception:
                pass

            p1_p2 = pr_pd[pr_pd["priority_tier"].isin(["P1", "P2"])].to_dict(orient="records")
            priority_data = {
                "all_regions":   pr_pd.to_dict(orient="records"),
                "p1_p2_regions": p1_p2,
                "total":         len(pr_pd),
                "p1_count":      int((pr_pd["priority_tier"] == "P1").sum()),
                "p2_count":      int((pr_pd["priority_tier"] == "P2").sum()),
            }
            if _run:
                mlflow.log_metric("priority_regions", len(pr_pd))
            step_msg = (f"Priority: {len(pr_pd)} regions | "
                        f"P1={priority_data['p1_count']} P2={priority_data['p2_count']}")
        except Exception as e:
            priority_data = {"error": str(e), "all_regions": [], "p1_p2_regions": []}
            step_msg      = f"Priority failed: {e}"

    payload   = _node_payload("priority_check", NodeStatus.SUCCESS, step_msg, 0.88,
                               priority_data.get("p1_p2_regions", [])[:5], [],
                               int((time.time() - t0) * 1000))
    remaining = [a for a in state.get("sub_agents", []) if a != "priority"]
    return {
        **state,
        "priority_data": priority_data,
        "sub_agents":    remaining,
        "steps":         [step_msg],
        "step_citations": state.get("step_citations", []) + [
            _step_cite("priority_node", state["query"], step_msg, [cfg.PRIORITY_TABLE], 0.88)
        ],
        "node_payloads": state.get("node_payloads", []) + [payload],
    }

# COMMAND ----------
# MAGIC %md ## 19. Medical Reasoning Node

# COMMAND ----------

MEDICAL_SYSTEM_PROMPT = """You are a senior medical advisor for sub-Saharan African healthcare.
Specialise in Ghana healthcare infrastructure analysis for the Virtue Foundation.

OUTPUT FORMAT:
1. CLINICAL INTERPRETATION (specific numbers + facility names)
2. CAPABILITY-GRAPH FINDINGS (what's missing in the dependency chain)
3. WORKFORCE / INFRASTRUCTURE GAP (region-level, specific)
4. MISREPRESENTATION RISK (which facilities, which claims, confidence level)
5. NGO RECOMMENDATIONS (3 concrete actions, named regions, urgency)

Cite specific facilities and regions. Never say "some facilities" without naming them.
Clinical rules: ICU + zero equipment = implausible; surgery + proc_count<2 + equip=0 = overstated."""


def medical_node(state: AgentStateV2) -> AgentStateV2:
    t0       = time.time()
    evidence: List[str] = []

    if state.get("sql_results"):
        try:
            rows = json.loads(state["sql_results"])
            if isinstance(rows, list) and rows and "error" not in rows[0]:
                evidence.append(
                    f"DATABASE ({len(rows)} rows):\n"
                    + json.dumps(rows[:15], indent=2, default=str)[:3000]
                )
        except Exception:
            evidence.append(f"SQL DATA: {str(state['sql_results'])[:2000]}")

    if state.get("anomaly_results"):
        evidence.append(
            f"ANOMALY FLAGS ({len(state['anomaly_results'])}):\n"
            + json.dumps(state["anomaly_results"][:8], indent=2, default=str)[:2500]
        )
    if state.get("graph_results"):
        evidence.append(
            f"CAPABILITY GRAPH ({len(state['graph_results'])} facilities):\n"
            + json.dumps(state["graph_results"][:8], indent=2, default=str)[:2500]
        )
    if state.get("geo_results") and not (state["geo_results"] or {}).get("error"):
        gr = state["geo_results"]
        evidence.append(
            f"GEOSPATIAL: {gr.get('specialty_matches',0)} facilities within "
            f"{gr.get('radius_km',50)}km of {gr.get('center',{}).get('name','?')}\n"
            + json.dumps(gr.get("facilities",[])[:8], indent=2, default=str)[:2000]
        )
    if state.get("desert_data"):
        dd = state["desert_data"]
        if dd.get("most_underserved"):
            evidence.append(
                "DESERT TOP 5:\n"
                + json.dumps(dd["most_underserved"][:5], indent=2, default=str)[:1800]
            )
    for r in (state.get("rag_results") or [])[:5]:
        m = r["metadata"]
        evidence.append(
            f"FACILITY [{m.get('name','?')}] "
            f"({m.get('facility_type','?')}, {m.get('region','?')}) "
            f"score={r['score']:.3f}\n{r['document'][:300]}"
        )

    if not evidence:
        evidence.append("No upstream data. Reasoning from general Ghana healthcare knowledge.")

    prompt = (
        f"Question: {state['query']}\n\n"
        f"Evidence:\n{'---'.join(evidence)[:5500]}\n\n"
        "Provide: (1) Clinical interpretation, (2) Capability-graph findings, "
        "(3) Workforce/infrastructure gaps, (4) Misrepresentation risks, "
        "(5) 3 concrete NGO actions."
    )
    analysis = call_llama(
        [{"role": "user", "content": prompt}],
        system_prompt=MEDICAL_SYSTEM_PROMPT,
        max_tokens=1400, temperature=0.2,
    )

    payload   = _node_payload("medical_reason", NodeStatus.SUCCESS,
                               f"Medical: {len(analysis)} chars", 0.75,
                               [{"analysis": analysis[:500]}], [],
                               int((time.time() - t0) * 1000))
    remaining = [a for a in state.get("sub_agents", []) if a != "medical"]
    return {
        **state,
        "medical_analysis": {"reasoning": analysis, "evidence_count": len(evidence)},
        "sub_agents":       remaining,
        "steps":            [f"Medical reasoning: {len(analysis)} chars"],
        "step_citations": state.get("step_citations", []) + [
            _step_cite("medical_node", f"{len(evidence)} evidence sources",
                       analysis[:200], ["llama_medical"], 0.75)
        ],
        "node_payloads": state.get("node_payloads", []) + [payload],
    }

# COMMAND ----------
# MAGIC %md ## 20. Planning Node

# COMMAND ----------

PLANNING_SYSTEM_PROMPT = """You are a healthcare planning advisor for the Virtue Foundation.
Audience: NGO programme officers — action-oriented, non-technical.

STRUCTURE:
A. IMMEDIATE (0–30 days): emergency gaps requiring urgent deployment
B. MEDIUM-TERM (1–3 months): staffing, equipment, training
C. LONG-TERM (6–12 months): structural investment, referral networks

For each action: Region + facility (if known) | Gap + population affected | Exact action | Impact metric | Cost tier."""


def planning_node(state: AgentStateV2) -> AgentStateV2:
    t0        = time.time()
    # FIX: initialise plan_text before the try block so it's always in scope
    plan_text = ""

    with _mlflow_nested("planning_node") as _run:
        if _run:
            mlflow.set_tag("step", "planning_node")
        try:
            pr_data    = state.get("priority_data") or {}
            pr_regions = pr_data.get("p1_p2_regions", pr_data.get("all_regions", []))[:10]

            reg_avail = get_table_cols(cfg.REGIONAL_TABLE)
            reg_sel   = [c for c in [
                "region_normalised", "total_facilities", "hospital_count",
                "ngo_count", "volunteer_facilities",
                "total_doctors", "total_beds",
                "icu_facilities", "emergency_medicine_facilities", "surgery_facilities",
                "missing_critical_specialties", "critical_specialty_gap_count",
                "medical_desert_score", "desert_label",
                "avg_ghost_probability", "avg_healthcare_maturity_score",
            ] if c in reg_avail]

            reg_pd = (
                spark.table(cfg.REGIONAL_TABLE)
                .select(*reg_sel)
                .orderBy(F.desc("medical_desert_score"))
                .toPandas()
                .infer_objects(copy=False)
            )
            for col in ["missing_critical_specialties"]:
                if col in reg_pd.columns:
                    reg_pd[col] = reg_pd[col].apply(ensure_list)

            plan_context = []
            for _, r in reg_pd.iterrows():
                rn     = safe_str(r.get("region_normalised", "?"))
                pr_row = next((p for p in pr_regions if p.get("region_normalised") == rn), {})
                plan_context.append({
                    "region":         rn,
                    "desert_label":   safe_str(r.get("desert_label", "?")),
                    "desert_score":   round(safe_float(r.get("medical_desert_score")), 3),
                    "priority_tier":  safe_str(pr_row.get("priority_tier", "?")),
                    "priority_score": round(safe_float(pr_row.get("regional_priority_score")), 3),
                    "total_facilities": safe_int(r.get("total_facilities")),
                    "hospitals":       safe_int(r.get("hospital_count")),
                    "ngos":            safe_int(r.get("ngo_count")),
                    "total_doctors":   safe_int(r.get("total_doctors")),
                    "total_beds":      safe_int(r.get("total_beds")),
                    "icu_facilities":  safe_int(r.get("icu_facilities")),
                    "emergency_fac":   safe_int(r.get("emergency_medicine_facilities")),
                    "surgery_fac":     safe_int(r.get("surgery_facilities")),
                    "missing_specs":   ensure_list(r.get("missing_critical_specialties"))[:5],
                    "interventions":   ensure_list(pr_row.get("recommended_interventions"))[:3],
                })

            prompt = (
                f"User question: {state['query']}\n\n"
                f"Healthcare data (sorted by desert severity + priority):\n"
                + json.dumps(plan_context, indent=2)[:5000]
                + "\n\nGenerate a specific numbered action plan with sections A/B/C."
            )
            plan_text = call_llama(
                [{"role": "user", "content": prompt}],
                system_prompt=PLANNING_SYSTEM_PROMPT,
                max_tokens=1500, temperature=0.2,
            )
            if _run:
                mlflow.log_text(plan_text, "action_plan.txt")
            step_msg = f"Planning: {len(plan_context)} regions, plan={len(plan_text)} chars"
        except Exception as e:
            step_msg = f"Planning failed: {e}"

    planning_data = {"plan": plan_text, "region_count": len(plan_context) if plan_text else 0}
    payload       = _node_payload("planning_sys", NodeStatus.SUCCESS, step_msg, 0.82,
                                   [{"plan": plan_text[:400]}] if plan_text else [], [],
                                   int((time.time() - t0) * 1000))
    remaining     = [a for a in state.get("sub_agents", []) if a != "planning"]
    return {
        **state,
        "planning_data": planning_data,
        "sub_agents":    remaining,
        "steps":         [step_msg],
        "step_citations": state.get("step_citations", []) + [
            _step_cite("planning_node", state["query"], plan_text[:200],
                       [cfg.REGIONAL_TABLE, cfg.PRIORITY_TABLE], 0.82)
        ],
        "node_payloads": state.get("node_payloads", []) + [payload],
    }

# COMMAND ----------
# MAGIC %md ## 21. NGO Node

# COMMAND ----------

def ngo_node(state: AgentStateV2) -> AgentStateV2:
    t0 = time.time()

    with _mlflow_nested("ngo_node") as _run:
        if _run:
            mlflow.set_tag("step", "ngo_node")
        try:
            avail   = get_table_cols(cfg.IDP_TABLE)
            ngo_sel = [c for c in [
                "name", "unique_id", "region_normalised", "city_clean",
                "ngo_serves_ghana", "accepts_volunteers_bool",
                "specialties_enriched", "capability_enriched",
                "data_completeness_score", "medical_desert_score", "desert_label",
                "has_emergency_medicine", "has_surgery",
                "service_richness_score", "healthcare_maturity_score", "is_planning_ready",
            ] if c in avail]

            ngo_pd = (
                spark.table(cfg.IDP_TABLE)
                .filter(F.col("is_ngo") == True)
                .select(*ngo_sel)
                .toPandas()
                .infer_objects(copy=False)
            )

            reg_avail = get_table_cols(cfg.REGIONAL_TABLE)
            reg_sel2  = [c for c in [
                "region_normalised", "ngo_count", "medical_desert_score",
                "desert_label", "missing_critical_specialties",
                "icu_facilities", "emergency_medicine_facilities",
            ] if c in reg_avail]

            regional_ngo = (
                spark.table(cfg.REGIONAL_TABLE)
                .select(*reg_sel2)
                .orderBy(F.desc("medical_desert_score"))
                .toPandas()
                .infer_objects(copy=False)
            )
            for col in ["missing_critical_specialties"]:
                if col in regional_ngo.columns:
                    regional_ngo[col] = regional_ngo[col].apply(ensure_list)

            ngo_gaps = regional_ngo[
                (pd.to_numeric(regional_ngo.get("ngo_count", pd.Series([0])),
                               errors="coerce").fillna(0) == 0) &
                (pd.to_numeric(regional_ngo.get("medical_desert_score", pd.Series([0])),
                               errors="coerce").fillna(0) > 0.45)
            ].to_dict(orient="records")

            ngo_data = {
                "total_ngos":        len(ngo_pd),
                "ngos":              ngo_pd.head(20).to_dict(orient="records"),
                "regional_coverage": regional_ngo.to_dict(orient="records"),
                "coverage_gaps":     ngo_gaps,
            }
            step_msg = (f"NGO: {len(ngo_pd)} NGOs | "
                        f"{len(ngo_gaps)} high-need regions without NGO coverage")
        except Exception as e:
            ngo_data = {"error": str(e), "total_ngos": 0, "coverage_gaps": []}
            step_msg = f"NGO failed: {e}"

    payload   = _node_payload("ngo_analysis", NodeStatus.SUCCESS, step_msg, 0.85,
                               ngo_data.get("coverage_gaps", [])[:5], [],
                               int((time.time() - t0) * 1000))
    remaining = [a for a in state.get("sub_agents", []) if a != "ngo"]
    return {
        **state,
        "ngo_data":  ngo_data,
        "sub_agents": remaining,
        "steps":     [step_msg],
        "node_payloads": state.get("node_payloads", []) + [payload],
    }

# COMMAND ----------
# MAGIC %md ## 22. Response Synthesiser

# COMMAND ----------

SYNTH_SYSTEM_PROMPT = """You are a healthcare intelligence analyst for the Virtue Foundation.
Audience: NGO programme officers — action-oriented, non-technical.

GROUND RULES:
- Always name specific facilities and regions from the data
- Use numbers: "7 of 17 regions lack X" not "several regions"
- Cite evidence: "According to facility records, [Name]..."
- Include confidence when uncertain: "(moderate confidence)"
- If map prepared: "An interactive map with N markers is ready"
- Never expose SQL, column names, JSON, or system internals
- Include recommended_actions / recommended_interventions when available"""


def synthesiser_node(state: AgentStateV2) -> AgentStateV2:
    t0       = time.time()
    evidence: List[str] = []
    facility_citations: List[Dict] = []

    if state.get("sql_results"):
        try:
            rows = json.loads(state["sql_results"])
            if isinstance(rows, list) and rows and "error" not in rows[0]:
                evidence.append(
                    f"DATA ({len(rows)} records):\n"
                    + json.dumps(rows[:20], indent=2, default=str)[:3000]
                )
        except Exception:
            evidence.append(f"DATA: {str(state['sql_results'])[:2000]}")

    for r in (state.get("rag_results") or [])[:6]:
        m    = r["metadata"]
        cits = r.get("citations", [])[:3]
        cit_str   = ("\n  SOURCE: " + " | ".join(c[:100] for c in cits if c)) if cits else ""
        graph_note= (f"\n  graph={r.get('graph_summary','')[:120]}"
                     f" | maturity={r.get('maturity',0):.2f}"
                     f" | svc_richness={r.get('svc_richness',0):.2f}")
        evidence.append(
            f"FACILITY [{m.get('name','?')}] "
            f"({m.get('facility_type','?')}, {m.get('city','?')}, "
            f"{m.get('region','?')}) relevance={r['score']:.3f}:\n"
            f"{r['document'][:350]}{cit_str}{graph_note}"
        )
        facility_citations.append({
            "facility_id":    m.get("unique_id", ""),
            "facility_name":  m.get("name", ""),
            "facility_type":  m.get("facility_type", ""),
            "region":         m.get("region", ""),
            "city":           m.get("city", ""),
            "relevance_score": float(r["score"]),
            "source_document": r["document"][:200],
            "idp_citations":   cits,
            "desert_label":    m.get("desert_label", ""),
            "svc_richness":    r.get("svc_richness", 0),
            "maturity":        r.get("maturity", 0),
            "step_name":       "rag_node",
        })

    if state.get("geo_results") and not (state["geo_results"] or {}).get("error"):
        gr = state["geo_results"]
        evidence.append(
            f"GEOSPATIAL: {gr.get('specialty_matches',0)} facilities within "
            f"{gr.get('radius_km',50)}km of {gr.get('center',{}).get('name','?').title()}\n"
            + json.dumps(gr.get("facilities",[])[:6], indent=2, default=str)[:2500]
        )
        if gr.get("cold_spots"):
            evidence.append("COLD SPOTS:\n"
                             + json.dumps(gr["cold_spots"], indent=2, default=str)[:1200])

    if state.get("anomaly_results"):
        evidence.append(
            f"ANOMALIES (top {min(len(state['anomaly_results']),6)}):\n"
            + json.dumps(state["anomaly_results"][:6], indent=2, default=str)[:2500]
        )
    if state.get("graph_results"):
        evidence.append(
            f"CAPABILITY GRAPH ({len(state['graph_results'])} findings):\n"
            + json.dumps(state["graph_results"][:6], indent=2, default=str)[:2500]
        )
    if state.get("desert_data"):
        dd = state["desert_data"]
        if dd.get("most_underserved"):
            evidence.append("MOST UNDERSERVED:\n"
                             + json.dumps(dd["most_underserved"][:5], indent=2, default=str)[:1800])
        if dd.get("cold_spots"):
            evidence.append("COLD SPOTS:\n"
                             + json.dumps(dd["cold_spots"][:5], indent=2, default=str)[:1200])
    if state.get("priority_data"):
        p1p2 = (state["priority_data"] or {}).get("p1_p2_regions", [])[:5]
        if p1p2:
            evidence.append("REGIONAL PRIORITIES (P1/P2):\n"
                             + json.dumps(p1p2, indent=2, default=str)[:2000])
    if (state.get("medical_analysis") or {}).get("reasoning"):
        evidence.append(f"CLINICAL ANALYSIS:\n{state['medical_analysis']['reasoning'][:2500]}")
    if (state.get("planning_data") or {}).get("plan"):
        evidence.append(f"NGO ACTION PLAN:\n{state['planning_data']['plan'][:2500]}")
    if (state.get("ngo_data") or {}).get("coverage_gaps"):
        evidence.append("NGO COVERAGE GAPS:\n"
                         + json.dumps(state["ngo_data"]["coverage_gaps"][:5], indent=2, default=str)[:1200])
    if state.get("map_data"):
        n_feat = len((state["map_data"] or {}).get("features", []))
        evidence.append(f"MAP: Interactive GeoJSON map prepared with {n_feat:,} markers.")

    if not evidence:
        evidence.append("No data retrieved. Drawing on general Ghana healthcare knowledge.")

    context = "\n\n---\n\n".join(evidence)
    synth   = (
        f'User question: "{state["query"]}"\n'
        f'Query type: {state.get("query_type","unknown")}\n\n'
        f"Evidence:\n{context[:7000]}"
    )

    answer = call_llama(
        [{"role": "user", "content": synth}],
        system_prompt=SYNTH_SYSTEM_PROMPT,
        max_tokens=1500, temperature=0.25,
    )

    conf, halluc, ret_qual = confidence_engine.compute(state)

    structured_answer = {
        "answer":             answer,
        "query_type":         state.get("query_type", ""),
        "confidence":         conf,
        "hallucination_risk": halluc,
        "retrieval_quality":  ret_qual,
        "evidence_sources":   len(evidence),
        "facility_citations": len(facility_citations),
        "map_ready":          state.get("map_data") is not None,
        "map_feature_count":  len((state.get("map_data") or {}).get("features", [])),
        "anomaly_count":      len(state.get("anomaly_results") or []),
        "desert_regions":     len((state.get("desert_data") or {}).get("all_regions", [])),
        "graph_findings":     len(state.get("graph_results") or []),
        "priority_p1p2":      len((state.get("priority_data") or {}).get("p1_p2_regions", [])),
    }

    with _mlflow_nested("synthesiser_node") as _run:
        if _run:
            mlflow.log_text(answer, "final_answer.txt")
            mlflow.log_metric("evidence_sources",   len(evidence))
            mlflow.log_metric("facility_citations", len(facility_citations))
            mlflow.log_metric("confidence",         conf)
            mlflow.log_metric("hallucination_risk", halluc)
            try:
                mlflow.log_dict({"citations": facility_citations}, "facility_citations.json")
            except Exception:
                pass

    payload = _node_payload(
        "synthesiser", NodeStatus.SUCCESS,
        f"Answer: {len(answer)} chars | conf={conf:.2f} | halluc={halluc:.2f}",
        conf, [], [], int((time.time() - t0) * 1000),
    )
    return {
        **state,
        "answer":            answer,
        "structured_answer": structured_answer,
        "citations":         facility_citations,
        "confidence_score":  conf,
        "hallucination_risk": halluc,
        "retrieval_quality": ret_qual,
        "steps":             ["Synthesiser: answer assembled"],
        "step_citations": state.get("step_citations", []) + [
            _step_cite("synthesiser", f"{len(evidence)} evidence sources",
                       answer[:200],
                       ["llama_3.3_70b"] + [c.get("facility_id", "") for c in facility_citations[:3]],
                       conf)
        ],
        "node_payloads": state.get("node_payloads", []) + [payload],
    }

# COMMAND ----------
# MAGIC %md ## 23. LangGraph Build

# COMMAND ----------

ALL_NODES = [
    "sql_query", "rag_search", "geo_calc", "map_gen",
    "anomaly_check", "capability_graph", "desert_check",
    "priority_check", "medical_reason", "planning_sys",
    "ngo_analysis", "synthesiser",
]

AGENT_TO_NODE: Dict[str, str] = {
    "sql":      "sql_query",
    "rag":      "rag_search",
    "geo":      "geo_calc",
    "map":      "map_gen",
    "anomaly":  "anomaly_check",
    "graph":    "capability_graph",
    "desert":   "desert_check",
    "priority": "priority_check",
    "medical":  "medical_reason",
    "planning": "planning_sys",
    "ngo":      "ngo_analysis",
    "general":  "rag_search",
}

PRIMARY_AGENT: Dict[str, str] = {
    "facility_lookup":           "rag_search",
    "facility_comparison":       "sql_query",
    "regional_analysis":         "sql_query",
    "desert_analysis":           "desert_check",
    "anomaly_analysis":          "anomaly_check",
    "capability_validation":     "capability_graph",
    "geo_search":                "geo_calc",
    "specialty_gap_analysis":    "sql_query",
    "healthcare_planning":       "planning_sys",
    "risk_assessment":           "anomaly_check",
    "infrastructure_analysis":   "capability_graph",
    "ngo_coordination":          "ngo_analysis",
    "capability_graph_reasoning":"capability_graph",
    "priority_analysis":         "priority_check",
    "general":                   "rag_search",
}

# All node names that _route_after_node can possibly return
_ALL_POSSIBLE_NEXT: Dict[str, str] = {n: n for n in ALL_NODES}


def _route_after_node(state: AgentStateV2) -> str:
    """
    FIX: Explicit guard so we ALWAYS return a key in _ALL_POSSIBLE_NEXT.
    If the agent name is unknown, route to synthesiser rather than raise ValueError.
    """
    remaining = state.get("sub_agents") or []
    if not remaining:
        return "synthesiser"
    nxt = remaining[0]
    node = AGENT_TO_NODE.get(nxt)
    # Guard: only return a node that's registered in ALL_NODES
    if node and node in _ALL_POSSIBLE_NEXT:
        return node
    return "synthesiser"


def build_virtue_agent():
    wf = StateGraph(AgentStateV2)

    wf.add_node("router",           router_node)
    wf.add_node("sql_query",        sql_node)
    wf.add_node("rag_search",       rag_node)
    wf.add_node("geo_calc",         geo_node)
    wf.add_node("map_gen",          map_node)
    wf.add_node("anomaly_check",    anomaly_node)
    wf.add_node("capability_graph", graph_node)
    wf.add_node("desert_check",     desert_node)
    wf.add_node("priority_check",   priority_node)
    wf.add_node("medical_reason",   medical_node)
    wf.add_node("planning_sys",     planning_node)
    wf.add_node("ngo_analysis",     ngo_node)
    wf.add_node("synthesiser",      synthesiser_node)

    wf.set_entry_point("router")

    def _route_from_router(state: AgentStateV2) -> str:
        node = PRIMARY_AGENT.get(state.get("query_type", ""), "rag_search")
        return node if node in _ALL_POSSIBLE_NEXT else "rag_search"

    # Router → primary node
    router_targets = {v: v for v in set(PRIMARY_AGENT.values()) if v in _ALL_POSSIBLE_NEXT}
    wf.add_conditional_edges("router", _route_from_router, router_targets)

    # Every work node → next sub-agent or synthesiser
    for node in ALL_NODES:
        if node != "synthesiser":
            wf.add_conditional_edges(node, _route_after_node, _ALL_POSSIBLE_NEXT)

    wf.add_edge("synthesiser", END)
    return wf.compile()


VIRTUE_AGENT = build_virtue_agent()
print(f"✅  LangGraph v5.1 compiled — {len(ALL_NODES)+2} nodes")
print(f"    Tables: {cfg.IDP_TABLE}")
print(f"            {cfg.FACILITIES_TABLE}")
print(f"            {cfg.REGIONAL_TABLE}")
print(f"            {cfg.DESERT_TABLE}")
print(f"            {cfg.ANOMALY_FLAGS}")
print(f"            {cfg.ANOMALY_REPORT}")
print(f"            {cfg.PRIORITY_TABLE}")

# COMMAND ----------
# MAGIC %md ## 24. Agent Runner (FIX: always returns a dict)

# COMMAND ----------

def _make_initial_state(query: str, session_id: str,
                        chat_history: Optional[List[Dict]],
                        run_id: str) -> AgentStateV2:
    """Return a fully-initialised AgentStateV2 with all fields set."""
    import uuid
    return AgentStateV2(
        query=query, session_id=session_id,
        query_id=str(uuid.uuid4())[:8],
        chat_history=chat_history or [],
        query_type="", query_complexity="moderate",
        execution_plan={}, sub_agents=[],
        sql_results=None, sql_row_count=0,
        rag_results=[], geo_results=None, map_data=None,
        anomaly_results=[], desert_top=[], desert_data=None,
        graph_results=[], priority_data=None,
        medical_analysis=None, planning_data=None, ngo_data=None,
        evidence_bundles=[], citations=[], step_citations=[],
        confidence_score=0.0, hallucination_risk=0.0, retrieval_quality=0.0,
        answer="", structured_answer={},
        warnings=[], errors=[], steps=[], node_payloads=[],
        run_id=run_id,
    )


def _make_error_output(query: str, error: str, run_id: str) -> Dict[str, Any]:
    """Return a valid output dict even when invoke fails completely."""
    return {
        "answer":           f"[Agent error — {error[:200]}]",
        "citations":        [], "step_citations": [], "steps":         [],
        "query_type":       "error", "query_complexity": "unknown",
        "sub_agents":       [], "execution_plan":   {}, "node_payloads": [],
        "sql_row_count":    0,  "sql_results":      None,
        "rag_results":      [], "geo_results":      None, "map_data":       None,
        "anomaly_results":  [], "desert_top":       [], "desert_data":     None,
        "graph_results":    [], "priority_data":    None, "medical_analysis": None,
        "planning_data":    None, "ngo_data":        None,
        "confidence_score": 0.0, "hallucination_risk": 1.0, "retrieval_quality": 0.0,
        "structured_answer": {"error": error},
        "warnings":         [error], "errors":           [error],
        "mlflow_run_id":    run_id,
    }


def run_agent(
    query: str,
    session_id: str = "demo",
    chat_history: Optional[List[Dict]] = None,
) -> Dict[str, Any]:
    """
    FIX: Always returns a Dict[str, Any], never None.
    The invoke call is wrapped in try/except so that any internal exception
    is caught and converted into an error output dict.
    """
    import uuid
    mlflow.set_experiment(cfg.MLFLOW_EXP)

    run_id  = "no-mlflow-run"
    output: Dict[str, Any] = {}

    try:
        with mlflow.start_run(run_name=f"v5_{query[:35].replace(' ','_')}") as run:
            run_id = run.info.run_id
            mlflow.set_tag("query",      query[:250])
            mlflow.set_tag("session_id", session_id)
            mlflow.set_tag("pipeline",   "virtue_langgraph_v5.1")

            initial = _make_initial_state(query, session_id, chat_history, run_id)

            # FIX: wrap invoke so any LangGraph exception is caught
            try:
                result = VIRTUE_AGENT.invoke(initial)
            except Exception as invoke_err:
                err_msg = f"invoke failed: {str(invoke_err)[:300]}"
                mlflow.set_tag("invoke_error", err_msg)
                return _make_error_output(query, err_msg, run_id)

            # result should be a dict; guard against None
            if result is None:
                return _make_error_output(query, "invoke returned None", run_id)

            output = {
                "answer":           result.get("answer", ""),
                "citations":        result.get("citations", []),
                "step_citations":   result.get("step_citations", []),
                "steps":            result.get("steps", []),
                "query_type":       result.get("query_type", ""),
                "query_complexity": result.get("query_complexity", ""),
                "sub_agents":       result.get("sub_agents", []),
                "execution_plan":   result.get("execution_plan", {}),
                "node_payloads":    result.get("node_payloads", []),
                "sql_row_count":    result.get("sql_row_count", 0),
                "sql_results":      result.get("sql_results"),
                "rag_results":      result.get("rag_results", []),
                "geo_results":      result.get("geo_results"),
                "map_data":         result.get("map_data"),
                "anomaly_results":  result.get("anomaly_results", []),
                "desert_top":       result.get("desert_top", []),
                "desert_data":      result.get("desert_data"),
                "graph_results":    result.get("graph_results", []),
                "priority_data":    result.get("priority_data"),
                "medical_analysis": result.get("medical_analysis"),
                "planning_data":    result.get("planning_data"),
                "ngo_data":         result.get("ngo_data"),
                "confidence_score":   result.get("confidence_score", 0),
                "hallucination_risk": result.get("hallucination_risk", 0),
                "retrieval_quality":  result.get("retrieval_quality", 0),
                "structured_answer":  result.get("structured_answer", {}),
                "warnings":    result.get("warnings", []),
                "errors":      result.get("errors", []),
                "mlflow_run_id": run_id,
            }

            mlflow.log_param("query_type",  output["query_type"])
            mlflow.log_metric("confidence", output["confidence_score"])
            mlflow.log_metric("halluc_risk",output["hallucination_risk"])
            mlflow.log_metric("citations",  len(output["citations"]))
            mlflow.log_metric("step_cits",  len(output["step_citations"]))
            try:
                mlflow.log_text(output["answer"], "final_answer.txt")
                mlflow.log_dict(output["structured_answer"], "structured_answer.json")
            except Exception:
                pass

    except Exception as outer_err:
        # mlflow.start_run itself failed or something outside invoke
        err_msg = f"run_agent outer error: {str(outer_err)[:300]}"
        return _make_error_output(query, err_msg, run_id)

    # Console summary
    print(f"\n{'='*70}")
    print(f"Query      : {query}")
    print(f"Type       : {output.get('query_type')} ({output.get('query_complexity')})")
    print(f"Plan       : {output.get('execution_plan',{}).get('execution_order','?')}")
    print(f"Steps      : {output.get('steps',[])}")
    print(f"Confidence : {output.get('confidence_score',0):.2f} | "
          f"Halluc risk: {output.get('hallucination_risk',0):.2f}")
    print(f"Citations  : {len(output.get('citations',[]))} fac | "
          f"{len(output.get('step_citations',[]))} nodes")
    if output.get("warnings"):
        print(f"Warnings   : {output['warnings'][:2]}")
    if output.get("errors"):
        print(f"Errors     : {output['errors'][:2]}")
    print(f"MLflow     : {run_id}")
    print(f"{'='*70}")
    print(f"\nAnswer:\n{output.get('answer','')}\n")
    return output

# COMMAND ----------
# MAGIC %md ## 25. Evaluation Suite

# COMMAND ----------

MUST_HAVE_QUERIES = [
    {"id": "1.1", "q": "How many hospitals have cardiology in Ghana?",               "expected": "regional_analysis"},
    {"id": "1.2", "q": "How many hospitals in Ashanti region have surgery?",         "expected": "regional_analysis"},
    {"id": "1.3", "q": "What services does Korle Bu Teaching Hospital offer?",       "expected": "facility_lookup"},
    {"id": "1.4", "q": "Are there any clinics in Kumasi that do dialysis?",          "expected": "facility_lookup"},
    {"id": "1.5", "q": "Which region in Ghana has the most hospitals?",              "expected": "regional_analysis"},
    {"id": "2.1", "q": "How many hospitals treating malaria are within 50km of Accra?","expected": "geo_search"},
    {"id": "2.2", "q": "Which facilities are within 30km of Tamale?",               "expected": "geo_search"},
    {"id": "2.3", "q": "Where are the largest geographic cold spots where surgery is absent?","expected": "geo_search"},
    {"id": "4.1", "q": "Which facilities have implausible ICU claims without infrastructure?","expected": "anomaly_analysis"},
    {"id": "4.2", "q": "Show facilities with enhanced ghost hospital flags",         "expected": "anomaly_analysis"},
    {"id": "4.4", "q": "Which facilities claim an unrealistic number of procedures?","expected": "anomaly_analysis"},
    {"id": "4.8", "q": "Which facilities have unusually high procedure breadth vs minimal infrastructure?","expected": "anomaly_analysis"},
    {"id": "4.9", "q": "Where do we see things that should not move together such as large bed counts with no surgical equipment?","expected": "anomaly_analysis"},
    {"id": "5.1", "q": "Which region in Ghana is the most medically underserved?",  "expected": "desert_analysis"},
    {"id": "5.2", "q": "Which regions are classified as Severe Desert?",            "expected": "desert_analysis"},
    {"id": "5.3", "q": "What specialties are missing from the top 5 underserved regions?","expected": "desert_analysis"},
    {"id": "5.4", "q": "Which regions have P1 or P2 intervention priority?",       "expected": "priority_analysis"},
    {"id": "5.5", "q": "What are the recommended interventions for Savannah region?","expected": "priority_analysis"},
    {"id": "6.1", "q": "Which facilities have critical capability dependency gaps?", "expected": "capability_graph_reasoning"},
    {"id": "6.2", "q": "Where is the surgical workforce actually practicing in Ghana?","expected": "regional_analysis"},
    {"id": "7.1", "q": "How many ICU-capable facilities exist per region?",         "expected": "regional_analysis"},
    {"id": "7.5", "q": "Which procedures depend on very few facilities in Ghana?",  "expected": "regional_analysis"},
    {"id": "7.6", "q": "Where is there oversupply of simple procedures vs scarcity of complex procedures?","expected": "regional_analysis"},
    {"id": "8.1", "q": "Generate an NGO action plan for the three most underserved regions","expected": "healthcare_planning"},
]


print("=" * 72)
print(f"VIRTUE FOUNDATION v5.1 EVALUATION SUITE ({len(MUST_HAVE_QUERIES)} queries)")
print("=" * 72)

eval_results = []
passed = 0

for item in MUST_HAVE_QUERIES:
    qid, query, expected = item["id"], item["q"], item["expected"]
    print(f"\n[{qid}] {query[:65]!r}")
    try:
        result = run_agent(query, session_id=f"eval_{qid}")

        # FIX: result is guaranteed non-None from run_agent, but guard anyway
        if not isinstance(result, dict):
            raise TypeError(f"run_agent returned {type(result).__name__} instead of dict")

        got_type = result.get("query_type", "unknown")
        type_ok  = (
            got_type == expected or
            expected in result.get("execution_plan", {}).get("execution_order", []) or
            result.get("execution_plan", {}).get("primary_agent", "") in
            {k for k, v in PRIMARY_AGENT.items() if v == PRIMARY_AGENT.get(expected, "")}
        )
        has_ans  = bool(result.get("answer") and len(result.get("answer","")) > 80)
        has_cits = (len(result.get("citations", [])) > 0 or
                    len(result.get("step_citations", [])) > 0)
        conf     = result.get("confidence_score", 0)
        halluc   = result.get("hallucination_risk", 0)

        if has_ans:
            passed += 1

        print(f"  Type     : {'✅' if type_ok else '⚠️'} got='{got_type}' expected='{expected}'")
        print(f"  Answer   : {'✅' if has_ans else '❌'} ({len(result.get('answer',''))} chars)")
        print(f"  Cits     : {len(result.get('citations',[]))} fac / "
              f"{len(result.get('step_citations',[]))} nodes")
        print(f"  Quality  : conf={conf:.2f} | halluc={halluc:.2f}")
        if result.get("warnings"):
            print(f"  Warnings : {result['warnings'][:2]}")
        if result.get("errors"):
            print(f"  Errors   : {result['errors'][:2]}")

        eval_results.append({
            "id": qid, "query": query,
            "expected": expected, "actual": got_type,
            "type_match": type_ok, "has_answer": has_ans,
            "citation_count": len(result.get("citations", [])),
            "confidence": conf, "hallucination_risk": halluc,
            "run_id": result.get("mlflow_run_id", ""),
        })
    except Exception as e:
        print(f"  ❌ ERROR: {e}")
        eval_results.append({
            "id": qid, "query": query,
            "expected": expected, "actual": "error",
            "error": str(e), "has_answer": False,
        })

print(f"\n{'='*72}")
print(f"RESULT: {passed}/{len(MUST_HAVE_QUERIES)} answered "
      f"({passed/len(MUST_HAVE_QUERIES)*100:.0f}%)")
print(f"{'='*72}")

try:
    mlflow.set_experiment(cfg.MLFLOW_EXP)
    with mlflow.start_run(run_name="06_eval_v5.1"):
        mlflow.log_metric("queries_answered", passed)
        mlflow.log_metric("pass_rate", passed / len(MUST_HAVE_QUERIES))
        mlflow.log_dict({"eval_results": eval_results}, "evaluation_v5.1.json")
    print("Evaluation logged to MLflow ✅")
except Exception as e:
    print(f"MLflow log skipped: {e}")

print()
print("✅  Notebook 06 v5.1 complete")
print(f"Usage:  result = run_agent('your question')")
print(f"        result = run_agent('question', chat_history=[...])")